# EE 3326 — Numerical and Scientific Data Analysis Using Python
## Final Group Project — VoltLab — Object-Oriented DC Circuit Simulator (Modified Nodal Analysis)

> **Build a mini-SPICE: read circuit netlists, solve any resistive DC circuit with Modified Nodal Analysis and NumPy linear algebra, and predict how resistor tolerances affect your design.**

| | |
|---|---|
| **Course** | EE 3326 — Numerical and Scientific Data Analysis Using Python |
| **Project type** | Group project — **5 students per team** |
| **Duration** | 3 weeks: **Thursday, November 19, 2026 → Thursday, December 10, 2026** |
| **Core skills tested** | Python basics · loops · decision making · functions · NumPy · Pandas · Object-Oriented Programming (OOP) · testing · Git/GitHub |
| **Application domain** | Circuit analysis — netlists, nodal analysis, linear algebra, component tolerances (Monte Carlo) |

### Project goal
Every circuit simulator — SPICE, LTspice, Multisim — reads a **netlist**, builds a system of linear equations with **Modified Nodal Analysis (MNA)**, and solves it. Your team will build **VoltLab**, a Python package that parses and cleans messy netlists (SI prefixes such as `4.7k`, `2mA`, lowercase types, comment lines), decodes resistor colour bands, represents every circuit element as an object, solves node voltages and branch currents with `numpy.linalg.solve`, verifies energy conservation, runs DC sweeps and RC step responses, and performs a **Monte Carlo tolerance analysis** to estimate manufacturing yield.

### Why this project is worth putting on your CV
When you finish, your team will have a public GitHub repository containing a documented, tested Python package that solves a real engineering problem — something you can show to a recruiter or talk about in an internship interview. Section **"Put this project on your CV"** at the end of this notebook explains how to present it.

## Project timeline & milestones

The project is graded through **four milestones (100 points)** plus **optional extra credit (up to 20 points)**, so the maximum score is **120 points**. Each milestone has its own section in this notebook with instructions, a checklist, and (where relevant) code cells you must complete.

| # | Milestone | What you submit | Due (11:59 PM) | Points |
|---|---|---|---|---|
| **M1** | Create the groups | Completed `TEAM` cell (Section M1) + team info form on the course LMS | **Fri, Nov 20, 2026** | 5 |
| **M2** | GitHub repository | Repo URL; all 5 members **and the instructor** added as collaborators; required folder structure pushed | **Tue, Nov 24, 2026** | 10 |
| **M3** | Working code & testing structure | All Parts A–F implemented, autograder tests passing, team `pytest` suite in `tests/`, GitHub release tag `v1.0-m3` | **Thu, Dec 3, 2026** | 55 |
| **M4** | Project video (8–10 min) & final presentation file | Video link + slide deck (`docs/presentation.pdf` or `.pptx`) + peer evaluation | **Thu, Dec 10, 2026** | 30 |
| **EC** | Extra credit *(optional)* | EC1 dashboard · EC2 advanced engineering feature · EC3 GitHub Actions CI · EC4 "go further" (see the *Extra Credit* section) | **Thu, Dec 10, 2026** | +20 |
| | | | **Total** | **100 + 20 EC = 120** |

**Suggested weekly plan**

| Week | Dates | Focus |
|---|---|---|
| Week 1 | Nov 19 – Nov 25 | M1 + M2 · Parts A & B (Python basics, functions, loops, decisions) · generate the dataset |
| Week 2 | Nov 26 – Dec 2 | Parts C, D & E (NumPy, Pandas, OOP) · write team tests as you go *(Thanksgiving week — plan ahead!)* |
| Week 3 | Dec 3 – Dec 10 | Part F integration · M3 release on Dec 3 · record video, build slides, polish README · M4 on Dec 10 |

### How to use this notebook
1. **Read the "Engineering background" section first.** It explains every formula you need. You do not need any prior knowledge of the application area.
2. Every Part starts with a **task table** (what each function must do) and a **📘 Concept review** with small runnable examples of the Python/NumPy/Pandas/OOP features you will need. Run the examples and change them — that is the fastest way to learn.
3. Inside every function you will find **`# HINT` comments with a step-by-step recipe** and a worked example. Delete `raise NotImplementedError()` and write your code under the hints.
4. Cells marked **`# PROVIDED`** already contain working code. Read them (you must be able to explain them in the video) but do not change them.
5. **Do not delete or rename** the functions, classes, or test cells. The autograder calls them by name.
6. Run the **test cell under each Part** — every test prints ✅ PASS, ❌ FAIL, 💥 ERROR, or ⬜ TODO. A failing test tells you which input gave which wrong answer.
7. Use only the libraries already imported (Python standard library, NumPy, Pandas, Matplotlib) unless a cell says otherwise. **In Part B you may *not* use NumPy or Pandas** — that part tests plain Python loops and decisions.
8. Keep the notebook runnable top-to-bottom: *Kernel → Restart & Run All* must work before every submission.

### Stuck? Try this checklist before asking for help
* Re-read the task table and the hint — most bugs are a missed requirement (e.g. *rounded to 2 decimals*, *largest first*, *return a new DataFrame*).
* Test the function by itself in a new cell with the example from the hint: `print(my_function(...))`.
* Add `print()` statements inside loops to see the values change, then remove them.
* Read the **last line** of an error message first — it names the problem (`KeyError: 'load_kw'` = a column name is wrong).
* Ask your teammates, then post on the course discussion board, then come to office hours.

---
# Engineering background — everything you need to know for VoltLab

This project uses only the circuit theory from your first circuits course. Here is a refresher of every idea you need.

### 1. Ohm's law, nodes and ground
* **Ohm's law:** `V = I × R`. A 1 kΩ resistor with 5 V across it carries 5 mA.
* A **node** is a point where components connect. We number nodes 0, 1, 2, … and **node 0 is ground** (0 V by definition). Every other node voltage is measured relative to ground.
* **Kirchhoff's Current Law (KCL):** the currents leaving a node add up to zero.
* **Power** absorbed by a component: `P = V × I` (also `V²/R` or `I²R` for a resistor).

### 2. Series, parallel and the voltage divider
* Series: `R = R1 + R2 + …`  · Parallel: `R = 1 / (1/R1 + 1/R2 + …)` (two 100 Ω in parallel = 50 Ω)
* **Voltage divider:** `V_out = V_in × R_bottom / (R_top + R_bottom)` → 12 V across 10 kΩ + 4.7 kΩ gives 12 × 4.7/14.7 = **3.84 V**.

### 3. Netlists and SI prefixes
A **netlist** is a text list of components: name, type, the two nodes, and the value. Engineers write values with SI prefixes: `4.7k` = 4,700 Ω, `2mA` = 0.002 A, `1M` = 1,000,000 Ω, `10u` = 0.000010. Capital `M` means *mega* but small `m` means *milli* — the case matters!

### 4. Resistor colour code
Through-hole resistors are labelled with coloured bands: **digit, digit, multiplier, tolerance**. *Yellow (4) – Violet (7) – Red (×100) – Gold (±5 %)* = 47 × 100 = **4.7 kΩ ± 5 %**. Real resistors only come in **standard values** (the E12 series: 1.0, 1.2, 1.5, 1.8, 2.2, 2.7, 3.3, 3.9, 4.7, 5.6, 6.8, 8.2 × powers of 10).

### 5. Nodal analysis — how a circuit simulator works
Write KCL at every node (except ground) using Ohm's law. Example: a 1 mA current source pushes current into node 1; R1 = 1 kΩ connects node 1 to node 2; R2 = 2 kΩ connects node 2 to ground:
```
node 1:  (V1 − V2)/1k            = 1 mA            →   [ 1/1k   −1/1k       ] [V1]   [1 mA]
node 2:  (V2 − V1)/1k + V2/2k    = 0                   [−1/1k    1/1k+1/2k  ] [V2] = [ 0  ]
```
This is the matrix equation **G · v = i**. NumPy solves it in one line: `np.linalg.solve(G, i)` → V1 = 3 V, V2 = 2 V.
Notice the pattern: a resistor between nodes a and b adds `+1/R` to the two diagonal entries `G[a,a]`, `G[b,b]` and `−1/R` to `G[a,b]`, `G[b,a]`. This pattern is called a **stamp** — each component "stamps" its own contribution into the matrix, so a simulator never needs to know the whole circuit at once. (Node *k* is row *k − 1* because ground has no row.)

**Voltage sources** need one extra unknown (their current) and one extra equation (`V_pos − V_neg = V`). This extension is called **Modified Nodal Analysis (MNA)** — it is what SPICE uses. The VoltageSource class is **provided** for you as a worked example.

### 6. RC circuits and the time constant
A capacitor C charging through R from 0 V towards V_s: `v(t) = V_s (1 − e^(−t/RC))`. After one **time constant τ = RC** it reaches **63.2 %** of V_s; after 5τ it is essentially fully charged.

### 7. Tolerances, Monte Carlo and yield
A "1 kΩ ± 5 %" resistor can be anything from 950 Ω to 1050 Ω. **Monte Carlo analysis** builds thousands of virtual circuits with random resistor values inside their tolerance and looks at the spread of the output. The **yield** is the percentage of circuits that meet the specification (e.g. output within ±2 % of nominal) — a key number in manufacturing.

### 8. What you will build
```
netlists.csv (messy) ──► Part D: clean ──► Part E: Circuit objects ◆── Resistor / VoltageSource / CurrentSource
                                                    │  solve() = stamp every component + np.linalg.solve
resistor_inventory.csv ──► Part D: decode colour codes, check stock
                                                    ▼
                         Part F: solve all circuits, summary table, menu  ──► video demo
  (Part A: unit helpers · Part B: colour codes, E12, netlist validation · Part C: matrix stamping, RC, Monte Carlo)
```

---
# Milestone 1 — Create the groups  *(due Fri, Nov 20, 2026 · 5 pts)*

Form a team of **exactly 5 students** and choose this project. Each member takes **one lead role**. A lead role means you *own* that area and answer questions about it in the video — it does **not** mean you only work on that area. **Every member must write code, write tests, and make commits.**

| Role | Primary responsibility | Leads |
|---|---|---|
| **Project Manager / Integration Lead** | Schedule, GitHub repo owner, merges pull requests, Part F integration & CLI | M2, Part F |
| **Data Engineer** | Data generation, loading, cleaning and aggregation with Pandas | Part D |
| **Numerical Methods Lead** | Vectorized NumPy algorithms and their validation | Part C |
| **Software Architect (OOP Lead)** | Class design, inheritance, composition, UML diagram | Part E |
| **Quality & Documentation Lead** | Team `pytest` suite, README, docstrings, video editing | Testing, M4 |

Parts **A** and **B** (Python basics, functions, loops, decisions) are split among **all five** members — each member implements at least one function from A or B.

**To complete Milestone 1:**
1. Fill in the `TEAM` dictionary below (one dictionary per member).
2. Run the cell — `check_team()` must print `Milestone 1 check: PASSED`.
3. Submit your team number, member names and roles on the course LMS.
4. Agree on a weekly meeting time and a communication channel (Discord, Teams, GroupMe…) and record them below.

In [ ]:
# ===================== MILESTONE 1: TEAM INFORMATION =====================
TEAM = {
    "team_number": None,                 # e.g. 7  (assigned by the instructor)
    "project": "VoltLab",
    "meeting_time": "",                  # e.g. "Mondays 6:00-7:00 PM, Library room 204"
    "communication_channel": "",         # e.g. "Discord server: ee3326-team7"
    "members": [
        {"name": "", "student_id": "", "email": "", "github": "", "role": "Project Manager / Integration Lead"},
        {"name": "", "student_id": "", "email": "", "github": "", "role": "Data Engineer"},
        {"name": "", "student_id": "", "email": "", "github": "", "role": "Numerical Methods Lead"},
        {"name": "", "student_id": "", "email": "", "github": "", "role": "Software Architect (OOP Lead)"},
        {"name": "", "student_id": "", "email": "", "github": "", "role": "Quality & Documentation Lead"},
    ],
}

REQUIRED_ROLES = {
    "Project Manager / Integration Lead", "Data Engineer", "Numerical Methods Lead",
    "Software Architect (OOP Lead)", "Quality & Documentation Lead",
}


def check_team(team):
    """Validate the TEAM dictionary and return a list of problems (empty list = OK)."""
    problems = []
    if not isinstance(team.get("team_number"), int):
        problems.append("team_number must be an integer")
    members = team.get("members", [])
    if len(members) != 5:
        problems.append(f"a team needs exactly 5 members (found {len(members)})")
    for i, m in enumerate(members, start=1):
        for field in ("name", "student_id", "email", "github", "role"):
            if not str(m.get(field, "")).strip():
                problems.append(f"member {i}: '{field}' is empty")
        if m.get("email") and "@" not in m["email"]:
            problems.append(f"member {i}: email looks invalid")
    roles = {m.get("role") for m in members}
    if roles != REQUIRED_ROLES:
        problems.append("each of the 5 lead roles must be assigned to exactly one member")
    for field in ("meeting_time", "communication_channel"):
        if not team.get(field, "").strip():
            problems.append(f"'{field}' is empty")
    print("Milestone 1 check:", "PASSED" if not problems else "NOT COMPLETE")
    for p in problems:
        print("  -", p)
    return problems


_ = check_team(TEAM)

---
# Milestone 2 — GitHub account & team repository  *(due Tue, Nov 24, 2026 · 10 pts)*

Professional engineers keep their code in version control. Your GitHub history is also how the instructor sees **who contributed what**, so commit early and often.

### Step 1 — Every member creates a GitHub account
* Sign up at <https://github.com> (or use your existing account). Use a **professional username** (e.g. `jsmith-ee`) — recruiters will see it.
* Add your real name and a profile photo; verify your e-mail. Students can also apply for the free **GitHub Student Developer Pack**.

### Step 2 — The Project Manager creates the team repository
* Repository name: **`ee3326-voltlab-teamNN`** (replace `NN` with your two-digit team number, e.g. `team07`).
* Keep it **Private** until final grades are posted (academic integrity). After grading you will make it **Public** for your CV.
* Initialize with a `README.md`, a Python `.gitignore`, and an MIT `LICENSE`.

### Step 3 — Add collaborators (all 4 teammates + the instructor)
*Settings → Collaborators → Add people*, then add:
* each of your **4 teammates**, and
* **the instructor: `INSTRUCTOR_GITHUB_USERNAME`** ← required for grading. *A repo the instructor cannot open earns 0 points for M2, M3 and M4.*

### Step 4 — Push the required folder structure
```
ee3326-voltlab-teamNN/
├── README.md                  # project title, team, how to install/run/test, results, video link
├── requirements.txt           # numpy, pandas, matplotlib, pytest
├── .gitignore
├── LICENSE
├── data/                      # generated CSV files (created by generate_dataset())
├── notebooks/
│   └── EE3326_Final_Project_P2_VoltLab.ipynb     # this notebook
├── src/
│   └── voltlab/
│       ├── __init__.py
│       ├── basics.py          # Parts A & B
│       ├── numerics.py        # Part C
│       ├── data_pipeline.py   # Part D
│       ├── models.py          # Part E
│       └── cli.py             # Part F
├── tests/
│   ├── test_basics.py
│   ├── test_numerics.py
│   ├── test_data_pipeline.py
│   └── test_models.py
└── docs/
    ├── uml_class_diagram.png
    └── presentation.pdf       # added for Milestone 4
```

### Step 5 — Agree on a Git workflow
* `main` is protected: **nobody pushes directly to `main`** after the initial setup.
* Each task gets a branch, e.g. `feature/part-c-numpy`, then a **Pull Request** reviewed and approved by one teammate before merging.
* Write meaningful commit messages (`Add moving_average with edge-case handling`, not `stuff`).
* Every member must have **at least 10 meaningful commits** by the end of the project (checked with `git shortlog -sn`).

**Submit for M2:** repo URL on the LMS + a screenshot of the *Collaborators* page showing all 5 members and the instructor. Then fill in and run the cell below.

In [ ]:
# ===================== MILESTONE 2: GITHUB REPOSITORY =====================
GITHUB = {
    "repo_url": "",        # e.g. "https://github.com/jsmith-ee/ee3326-voltlab-team07"
    "instructor_added": False,   # set to True after adding INSTRUCTOR_GITHUB_USERNAME as a collaborator
}


def check_github(github, team):
    """Validate the GITHUB dictionary against the TEAM dictionary."""
    import re
    problems = []
    url = github.get("repo_url", "").strip()
    if not re.fullmatch(r"https://github\.com/[\w.-]+/ee3326-voltlab-team\d{2}/?", url):
        problems.append("repo_url must look like https://github.com/<owner>/ee3326-voltlab-teamNN")
    if not github.get("instructor_added"):
        problems.append("add the instructor (INSTRUCTOR_GITHUB_USERNAME) as a collaborator, then set instructor_added=True")
    usernames = [m.get("github", "").strip() for m in team["members"]]
    if any(not u for u in usernames):
        problems.append("every member needs a GitHub username in TEAM")
    if len(set(u.lower() for u in usernames if u)) != len([u for u in usernames if u]):
        problems.append("GitHub usernames must be unique")
    print("Milestone 2 check:", "PASSED" if not problems else "NOT COMPLETE")
    for p in problems:
        print("  -", p)
    return problems


_ = check_github(GITHUB, TEAM)

---
# Milestone 3 — Working code & testing structure  *(due Thu, Dec 3, 2026 · 55 pts)*

Milestone 3 is the engineering core of the project. It is organised in six parts that build on each other:

| Part | Topic | Skills tested | Autograder pts |
|---|---|---|---|
| **A** | Engineering helper functions | variables, arithmetic, type checks, functions, default arguments, raising exceptions | 5 |
| **B** | Loops & decision making | `for`/`while` loops, `if/elif/else`, lists, dictionaries — **no NumPy/Pandas** | 6 |
| **C** | Numerical analysis | NumPy arrays, vectorization, boolean masks, `np.polyfit`, statistics | 7 |
| **D** | Data pipeline | Pandas: read CSV, clean, `groupby`, `merge`, time series | 7 |
| **E** | Object-oriented model | classes, `__init__`, abstract base classes, inheritance, polymorphism, composition, dunder methods | 7 |
| **F** | Integration, report & CLI | putting everything together, Matplotlib, menu-driven program | 3 |
| **T** | Team test suite (`pytest`) | writing your own unit tests in `tests/` | 10 |
| **Q** | Code quality & Git history | docstrings, comments, PRs, commits from every member | 10 |
| **EC** | *Extra credit (optional)* | dashboard, advanced feature, CI, going further | +20 |

### Testing structure (read this first)
Your work is tested at **two levels**:
1. **Provided autograder tests** — the cell at the end of every Part runs tests written by the instructor. Do **not** modify them. The instructor will also run a *hidden* test set with different numbers, so do not hard-code answers.
2. **Your team's `pytest` suite** — in your repository's `tests/` folder (see *Part T*). You write these yourselves: at least **3 tests per member (≥ 15 total)**, covering normal cases, edge cases, and error cases.

**Recommended order of work:** A → B → C → D → E → F. Each Part uses functions from the earlier Parts (for example, the Pandas pipeline in Part D calls your Part A and Part C functions, and the classes in Part E call functions from Parts B and C). Split the work so that several people can start in parallel: Parts A, B and C do not depend on each other.

Run the setup cell below first: it imports the libraries and defines the small testing framework used by every Part. *(You do not need to understand the testing framework's code, but it is a nice example of functions, loops, `try/except` and dictionaries.)*

In [ ]:
# ===================== SETUP: imports + mini test framework =====================
import math
import random
from abc import ABC, abstractmethod
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

_SCORES = {}   # part -> (earned, possible)


def run_tests(part, tests):
    """Run a list of (name, test_function, points) tuples and record the score for `part`."""
    earned = possible = 0
    print(f"========== Part {part} tests ==========")
    for name, fn, pts in tests:
        possible += pts
        try:
            fn()
        except NotImplementedError:
            print(f"⬜ TODO   {name}  (not implemented yet)")
        except AssertionError as e:
            print(f"❌ FAIL   {name}  -> {e}")
        except Exception as e:  # noqa: BLE001 - we want to report every error
            print(f"💥 ERROR  {name}  -> {type(e).__name__}: {e}")
        else:
            earned += pts
            print(f"✅ PASS   {name}  (+{pts})")
    _SCORES[part] = (earned, possible)
    print(f"---------- Part {part}: {earned}/{possible} points ----------")


def assert_close(actual, expected, tol=1e-6, msg=""):
    """Assert two numbers (or arrays) are equal within a tolerance."""
    assert np.allclose(actual, expected, atol=tol, rtol=1e-6), f"{msg} expected {expected}, got {actual}"


def assert_raises(exc_type, fn, *args, **kwargs):
    """Assert that calling fn(*args, **kwargs) raises exc_type."""
    try:
        fn(*args, **kwargs)
    except NotImplementedError:
        raise
    except exc_type:
        return
    except Exception as e:  # noqa: BLE001
        raise AssertionError(f"expected {exc_type.__name__}, got {type(e).__name__}") from e
    raise AssertionError(f"expected {exc_type.__name__} to be raised")


def try_demo(fn):
    """Run a demo function; print a friendly message if a part is not finished yet."""
    try:
        return fn()
    except NotImplementedError:
        print("⬜ Demo skipped — finish the functions/classes this demo uses first.")


print("Setup complete. NumPy", np.__version__, "| Pandas", pd.__version__)

## Dataset — generate the netlists (provided, do not modify)

**`data/netlists.csv`** — five circuits written by different (careless) engineers:

| column | description |
|---|---|
| `circuit_id` | `C1` … `C5` (rows starting with `#` are comments) |
| `name` | component name, e.g. `R1`, `V1`, `I1` |
| `type` | `R` resistor, `V` voltage source, `I` current source — sometimes lowercase or padded with spaces |
| `node_pos`, `node_neg` | node numbers; node **0 is ground**. Sometimes written as `"2.0"` |
| `value` | text with SI prefixes / units: `"4.7k"`, `"10 K"`, `"2.2kΩ"`, `"12V"`, `"2mA"`, `"1M"` … |
| `tolerance_pct` | resistor tolerance in % (blank means the default 5 %) |

| Circuit | Description |
|---|---|
| C1 | Voltage divider (12 V, 10 kΩ, 4.7 kΩ) |
| C2 | Wheatstone bridge |
| C3 | Three-section R-2R ladder (each section halves the voltage) |
| C4 | Current source feeding a resistor network |
| C5 | Two voltage sources (superposition check) |

**`data/resistor_inventory.csv`** — the lab's parts drawer: `part_id, bands, quantity, unit_cost_usd`, where `bands` is a colour code such as `"yellow-violet-red-gold"`.

In [ ]:
def generate_dataset(data_dir=DATA_DIR):
    """Write data/netlists.csv and data/resistor_inventory.csv (deliberately messy)."""
    rows = [
        # circuit_id, name, type, node_pos, node_neg, value, tolerance_pct
        ("C1", "V1", "V", "1", "0", "12V", ""), ("C1", "R1", "R", "1", "2", "10k", "1"),
        ("C1", "R2", " r ", "2", "0", "4.7 k", ""),
        ("# Wheatstone bridge", "", "", "", "", "", ""),
        ("C2", "V1", "V", "1", "0", "5", ""), ("C2", "R1", "R", "1", "2", "1k", ""),
        ("C2", "R2", "R", "1", "3", "2.2kΩ", ""), ("C2", "R3", "r", "2", "0", "2.2K", ""),
        ("C2", "R4", "R", "3", "0", "1000", ""), ("C2", "R5", "R", "2", "3", "10k", "10"),
        ("C3", "V1", "v", "1", "0", "9 V", ""), ("C3", "R1", "R", "1", "2", "1k", "1"),
        ("C3", "R2", "R", "2", "0", "2k", "1"), ("C3", "R3", "R", "2", "3", "1k", "1"),
        ("C3", "R4", "R", "3", "0", "2k", "1"), ("C3", "R5", "R", "3", "4", "1k", "1"),
        ("C3", "R6", "R", "4.0", "0", "2k", "1"), ("C3", "R7", "R", "4", "0", "2000", "1"),
        ("C4", "I1", "I", "1", "0", "2mA", ""), ("C4", "R1", "R", "1", "0", "1k", ""),
        ("C4", "R2", "R", "1", "2", "3.3k", ""), ("C4", "R3", "R", "2", "0", "6.8k", ""),
        ("C5", "V1", "V", "1", "0", "10", ""), ("C5", "V2", "V", "3", "0", "5V", ""),
        ("C5", "R1", "R", "1", "2", "1k", ""), ("C5", "R2", "R", "2", "3", "2k", ""),
        ("C5", "R3", "R", "2", "0", "3.3k", ""), ("C5", "RX", "R", "2", "0", "abc", ""),
    ]
    rows.insert(15, rows[14])                       # duplicated line in C3
    cols = ["circuit_id", "name", "type", "node_pos", "node_neg", "value", "tolerance_pct"]
    pd.DataFrame(rows, columns=cols).to_csv(data_dir / "netlists.csv", index=False)
    inventory = pd.DataFrame({
        "part_id": [f"P{i:03d}" for i in range(1, 12)],
        "bands": ["brown-black-orange-gold", "yellow-violet-red-gold", "red-red-red-gold",
                  "brown-black-red-brown", "brown-black-red-gold", "red-black-red-gold",
                  "orange-orange-red-gold", "brown-black-brown-gold", "brown-black-yellow-silver",
                  "green-blue-orange", "Blue-Grey-Red-Gold"],
        "quantity": [120, 45, 60, 30, 200, 0, 80, 150, 25, 10, 40],
        "unit_cost_usd": [0.02, 0.02, 0.02, 0.08, 0.02, 0.02, 0.02, 0.02, 0.03, 0.05, 0.02],
    })
    inventory.to_csv(data_dir / "resistor_inventory.csv", index=False)
    return data_dir / "netlists.csv", data_dir / "resistor_inventory.csv"


NETLIST_CSV, INVENTORY_CSV = generate_dataset()
print(pd.read_csv(NETLIST_CSV, dtype=str).to_string())

---
## Part A — Circuit helper functions  *(Python basics, strings & functions · 5 pts)*

| Function | Specification |
|---|---|
| `parse_si_value(text)` | Convert a value with an SI prefix to a float. Numbers (`int`/`float`) are returned as `float`. For strings: remove all spaces; remove **one** trailing unit if present (`"ohm"`, `"Ω"`, `"V"`, `"A"`, `"F"`); then if the last character is a prefix multiply: `p`=1e-12, `n`=1e-9, `u`/`µ`=1e-6, `m`=1e-3, `k`/`K`=1e3, `M`=1e6, `G`=1e9. Examples: `"4.7k"`→4700, `"2mA"`→0.002, `"1M"`→1e6, `"10 uF"`→1e-5. Raise `ValueError` if the text cannot be parsed. |
| `series_resistance(resistances)` | Sum of a list of resistances. Raise `ValueError` if the list is empty or any value ≤ 0. |
| `parallel_resistance(resistances)` | `1 / Σ(1/Rᵢ)`, same validation. |
| `voltage_divider(v_in, r_top, r_bottom)` | Output voltage of a two-resistor divider. |
| `power_w(voltage=None, current=None, resistance=None)` | Power from **any two** of the three quantities (`V·I`, `V²/R` or `I²R`). Raise `ValueError` if fewer than two are given. |

### 📘 Concept review for Part A — writing good functions

A function takes **inputs (parameters)**, does a calculation, and **returns** an output. Good engineering functions also **check their inputs** and **raise an error** when the input is physically impossible (negative time, zero capacity…), instead of silently returning a wrong number.

| Feature | Syntax | Notes |
|---|---|---|
| Define a function | `def name(a, b):` | the body is indented 4 spaces |
| Default argument | `def name(a, b=1.0):` | callers may omit `b` |
| Docstring | `"""One line saying what it returns."""` | first line inside the function |
| Return a value | `return a * b` | the function stops here |
| Raise an error | `raise ValueError("message")` | use for impossible inputs |
| Round | `round(x, 2)` | 2 decimal places |
| Min / max of two numbers | `min(a, b)`, `max(a, b)` | handy to "cap" or "clamp" a value |

**Testing for errors:** the test `assert_raises(ValueError, f, 1, -2)` passes only if calling `f(1, -2)` raises a `ValueError`.

Run the example below, then try changing the inputs.

In [ ]:
# EXAMPLE (not part of the project) — a validated function with a default argument
def rectangle_area(width, height=1.0):
    """Return the area of a rectangle; raise ValueError for negative sizes."""
    if width < 0 or height < 0:
        raise ValueError("sizes must be non-negative")
    return width * height


print(rectangle_area(3, 4))        # 12
print(rectangle_area(5))           # 5.0  -> height used its default value
print(round(2.34567, 2))           # 2.35
print(min(150.0, 100.0))           # 100.0 -> "cap" a percentage at 100
try:
    rectangle_area(-1, 2)
except ValueError as err:          # this is what assert_raises checks for you
    print("Caught the error:", err)

In [ ]:
SI_PREFIXES = {"p": 1e-12, "n": 1e-9, "u": 1e-6, "µ": 1e-6, "m": 1e-3,
               "k": 1e3, "K": 1e3, "M": 1e6, "G": 1e9}
UNITS = ("ohm", "Ω", "V", "A", "F")


def parse_si_value(text):
    """Convert a number or a string such as '4.7k' or '2mA' to a float in base SI units."""
    # HINT ── step by step:
    #   1. If text is already a number (isinstance(text, (int, float)) and not a bool): return float(text)
    #   2. s = str(text).replace(" ", "")                         -> "4.7 k" becomes "4.7k"
    #   3. for unit in UNITS: if s.endswith(unit) and len(s) > len(unit): s = s[:-len(unit)]; break
    #   4. multiplier = 1.0; if s and s[-1] in SI_PREFIXES: multiplier = SI_PREFIXES[s[-1]]; s = s[:-1]
    #   5. try: return float(s) * multiplier
    #      except ValueError: raise ValueError(f"cannot parse value {text!r}")
    #   Example: "2mA" -> remove "A" -> "2m" -> 2 * 1e-3 = 0.002
    # YOUR CODE HERE
    raise NotImplementedError()


def series_resistance(resistances):
    """Return the equivalent resistance of resistors in series."""
    # HINT ── 1. if len(resistances) == 0 or any(r <= 0 for r in resistances): raise ValueError(...)
    #   2. return float(sum(resistances))
    # YOUR CODE HERE
    raise NotImplementedError()


def parallel_resistance(resistances):
    """Return the equivalent resistance of resistors in parallel."""
    # HINT ── same validation as series_resistance, then
    #   return 1.0 / sum(1.0 / r for r in resistances)
    #   Example: parallel_resistance([100, 100]) -> 50.0
    # YOUR CODE HERE
    raise NotImplementedError()


def voltage_divider(v_in, r_top, r_bottom):
    """Return the output voltage across r_bottom."""
    # HINT ── return v_in * r_bottom / (r_top + r_bottom)
    # YOUR CODE HERE
    raise NotImplementedError()


def power_w(voltage=None, current=None, resistance=None):
    """Return the power in watts from any two of voltage, current and resistance."""
    # HINT ── an if/elif chain that checks which arguments are not None:
    #   if voltage is not None and current is not None: return voltage * current
    #   elif voltage is not None and resistance is not None: return voltage ** 2 / resistance
    #   elif current is not None and resistance is not None: return current ** 2 * resistance
    #   else: raise ValueError(...)
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
def _test_a1():
    cases = {"4.7k": 4700, "2mA": 0.002, "1M": 1e6, "10 uF": 1e-5, "2.2kΩ": 2200, "10 K": 1e4,
             "12V": 12, "100": 100, "1e3": 1000, "470n": 4.7e-7, "3.3ohm": 3.3}
    for text, expected in cases.items():
        assert_close(parse_si_value(text), expected, tol=1e-12, msg=text)
    assert parse_si_value(5) == 5.0 and isinstance(parse_si_value(5), float)

def _test_a2():
    for bad in ["abc", "", "k", "4.7x"]:
        assert_raises(ValueError, parse_si_value, bad)

def _test_a3():
    assert_close(series_resistance([100, 220, 330]), 650)
    assert_close(parallel_resistance([100, 100]), 50)
    assert_close(parallel_resistance([1000, 2000, 2000]), 500)
    assert_raises(ValueError, series_resistance, [])
    assert_raises(ValueError, parallel_resistance, [100, 0])

def _test_a4():
    assert_close(voltage_divider(12, 10_000, 4_700), 12 * 4.7 / 14.7)
    assert_close(voltage_divider(5, 1, 1), 2.5)

def _test_a5():
    assert_close(power_w(voltage=5, current=0.1), 0.5)
    assert_close(power_w(voltage=10, resistance=100), 1.0)
    assert_close(power_w(current=0.02, resistance=1000), 0.4)
    assert_raises(ValueError, power_w, voltage=3)

run_tests("A", [
    ("parse_si_value prefixes and units", _test_a1, 1),
    ("parse_si_value invalid text", _test_a2, 1),
    ("series_resistance / parallel_resistance", _test_a3, 1),
    ("voltage_divider", _test_a4, 1),
    ("power_w with default arguments", _test_a5, 1),
])

---
## Part B — Loops & decision making  *(6 pts — **do not use NumPy or Pandas here**)*

**Resistor colour code.** Digits: black 0, brown 1, red 2, orange 3, yellow 4, green 5, blue 6, violet 7, grey 8, white 9. The multiplier band may also be gold (×0.1) or silver (×0.01). Tolerance band: brown 1 %, red 2 %, gold 5 %, silver 10 %; a 3-band resistor has 20 % tolerance.

**E12 standard values:** 1.0, 1.2, 1.5, 1.8, 2.2, 2.7, 3.3, 3.9, 4.7, 5.6, 6.8, 8.2 (× any power of 10).

| Function | Specification |
|---|---|
| `color_code_to_ohms(bands)` | `bands` is a string like `"Yellow-Violet-Red-Gold"` (case-insensitive, `-` separated). Return a tuple `(ohms, tolerance_pct)`. Raise `ValueError` for an unknown colour or if there are not 3 or 4 bands. |
| `nearest_e12(value)` | Return the E12 standard value closest to `value` (search the decade of `value` and its neighbours with loops; on a tie choose the smaller value). Raise `ValueError` for `value <= 0`. |
| `validate_netlist(rows)` | `rows` is a list of dicts with keys `name, type, node_pos, node_neg, value`. Return a **list of error strings** (empty list = valid). Each message must contain the component name. Detect: unknown type (not R/V/I) · both terminals on the same node · resistor value ≤ 0 · duplicate component name. Also add the message `"no ground (node 0) in circuit"` if no component touches node 0. |

### 📘 Concept review for Part B — loops and decisions

Almost every Part B function is one of these **five loop patterns**. Recognise the pattern first, then write the code.

| Pattern | Idea | Skeleton |
|---|---|---|
| **Accumulate** | add up values | `total = 0` → `for v in values: total += v` |
| **Count** | count values that satisfy a condition | `count = 0` → `for v in values: if cond: count += 1` |
| **Classify** | pick one label from ranges | `if x < 10: ... elif x < 20: ... else: ...` (check the ranges **in order**) |
| **Group / tally** | count per category | `counts = {"a": 0, "b": 0}` → `counts[label] += 1` |
| **Streak / run** | longest run of consecutive values | keep `current` and `best`; reset `current = 0` when the run breaks |

Useful helpers: `enumerate(values)` gives `(index, value)` pairs, `zip(a, b)` walks two lists together, `return` inside a loop stops early (e.g. "first value that…"), and `isinstance(x, int)` checks a type.

**Boundary values matter:** the tests always check the exact edges (e.g. is 60 "Normal" or "Bradycardia"?). Read whether the task says `<` or `<=`.

In [ ]:
# EXAMPLES (not part of the project) — the five patterns on a list of temperatures
temps = [18, 22, 31, 33, 35, 24, 36, 37, 20]

total = 0                                   # 1) accumulate
for t in temps:
    total += t
print("average:", total / len(temps))

hot = 0                                     # 2) count
for t in temps:
    if t > 30:
        hot += 1
print("hot readings:", hot)

def comfort(t):                             # 3) classify (order of the checks matters!)
    if t < 20:
        return "cold"
    elif t <= 26:
        return "comfortable"
    else:
        return "hot"

tally = {"cold": 0, "comfortable": 0, "hot": 0}   # 4) group / tally
for t in temps:
    tally[comfort(t)] += 1
print(tally)

best = current = 0                          # 5) longest streak above 30
for t in temps:
    if t > 30:
        current += 1
        best = max(best, current)
    else:
        current = 0
print("longest hot streak:", best)

for i, t in enumerate(temps):               # "first index where ..." -> return/break early
    if t > 34:
        print("first reading above 34 °C is at index", i)
        break

In [ ]:
DIGITS = {"black": 0, "brown": 1, "red": 2, "orange": 3, "yellow": 4,
          "green": 5, "blue": 6, "violet": 7, "grey": 8, "white": 9}
MULTIPLIERS = {**{c: 10 ** d for c, d in DIGITS.items()}, "gold": 0.1, "silver": 0.01}
TOLERANCES = {"brown": 1.0, "red": 2.0, "gold": 5.0, "silver": 10.0}
E12 = [1.0, 1.2, 1.5, 1.8, 2.2, 2.7, 3.3, 3.9, 4.7, 5.6, 6.8, 8.2]


def color_code_to_ohms(bands):
    """Decode a 3- or 4-band resistor colour code into (ohms, tolerance_pct)."""
    # HINT ── step by step:
    #   1. colors = [c.strip().lower() for c in bands.split("-")]
    #   2. if len(colors) not in (3, 4): raise ValueError(...)
    #   3. if colors[0] or colors[1] not in DIGITS, or colors[2] not in MULTIPLIERS: raise ValueError(...)
    #   4. ohms = (DIGITS[colors[0]] * 10 + DIGITS[colors[1]]) * MULTIPLIERS[colors[2]]
    #   5. tolerance: 20.0 if there are 3 bands; else TOLERANCES[colors[3]] (ValueError if unknown)
    #   6. return (ohms, tolerance)
    #   Example: "yellow-violet-red-gold" -> (47 * 100, 5.0) = (4700, 5.0)
    # YOUR CODE HERE
    raise NotImplementedError()


def nearest_e12(value):
    """Return the closest E12 standard resistor value."""
    # HINT ── step by step:
    #   1. if value <= 0: raise ValueError(...)
    #   2. decade = math.floor(math.log10(value))          -> 5000 gives 3
    #   3. best = None
    #   4. for d in (decade - 1, decade, decade + 1):        (neighbouring decades: 95 is closest to 100!)
    #          for base in E12:
    #              candidate = base * 10 ** d
    #              if best is None or abs(candidate - value) < abs(best - value): best = candidate
    #   5. return round(best, 9)                          (removes round-off such as 4700.000000000001)
    #   Example: nearest_e12(5000) -> 4700 (4700 is 300 away, 5600 is 600 away)
    # YOUR CODE HERE
    raise NotImplementedError()


def validate_netlist(rows):
    """Return a list of human-readable problems found in a netlist (empty list = OK)."""
    # HINT ── step by step:
    #   1. errors = []; seen = set(); has_ground = False
    #   2. for row in rows: name = row["name"]
    #        if name in seen: errors.append(f"{name}: duplicate component name");   seen.add(name)
    #        if row["type"] not in ("R", "V", "I"): errors.append(f"{name}: unknown type ...")
    #        if row["node_pos"] == row["node_neg"]: errors.append(f"{name}: both terminals on the same node")
    #        if row["type"] == "R" and row["value"] <= 0: errors.append(f"{name}: resistance must be positive")
    #        if row["node_pos"] == 0 or row["node_neg"] == 0: has_ground = True
    #   3. after the loop: if not has_ground: errors.append("no ground (node 0) in circuit")
    #   4. return errors
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
def _test_b1():
    assert_close(color_code_to_ohms("yellow-violet-red-gold")[0], 4700)
    assert_close(color_code_to_ohms("brown-black-orange-gold")[0], 10_000)
    assert_close(color_code_to_ohms("Blue-Grey-Red-Gold")[0], 6800)
    assert_close(color_code_to_ohms("yellow-violet-gold-gold")[0], 4.7)

def _test_b2():
    assert color_code_to_ohms("brown-black-red-brown")[1] == 1.0
    assert color_code_to_ohms("brown-black-yellow-silver")[1] == 10.0
    ohms, tol = color_code_to_ohms("green-blue-orange")
    assert_close(ohms, 56_000); assert tol == 20.0

def _test_b3():
    assert_raises(ValueError, color_code_to_ohms, "pink-black-red-gold")
    assert_raises(ValueError, color_code_to_ohms, "brown-black")
    assert_raises(ValueError, color_code_to_ohms, "brown-black-red-blue-gold-red")

def _test_b4():
    for value, expected in [(5000, 4700), (5300, 5600), (1000, 1000), (95, 100), (0.013, 0.012), (9.0e5, 8.2e5)]:
        assert_close(nearest_e12(value), expected, tol=1e-9, msg=f"nearest_e12({value})")
    assert_raises(ValueError, nearest_e12, 0)

def _test_b5():
    good = [{"name": "V1", "type": "V", "node_pos": 1, "node_neg": 0, "value": 5.0},
            {"name": "R1", "type": "R", "node_pos": 1, "node_neg": 0, "value": 1e3}]
    assert validate_netlist(good) == []

def _test_b6():
    bad = [{"name": "V1", "type": "V", "node_pos": 1, "node_neg": 2, "value": 5.0},
           {"name": "R1", "type": "R", "node_pos": 1, "node_neg": 1, "value": 1e3},
           {"name": "R2", "type": "R", "node_pos": 1, "node_neg": 2, "value": -5},
           {"name": "R2", "type": "R", "node_pos": 2, "node_neg": 3, "value": 10},
           {"name": "L1", "type": "L", "node_pos": 3, "node_neg": 1, "value": 1e-3}]
    errs = validate_netlist(bad)
    assert len(errs) == 5, errs
    assert any("R1" in e for e in errs) and any("L1" in e for e in errs)
    assert sum("R2" in e for e in errs) == 2
    assert any("ground" in e for e in errs)

run_tests("B", [
    ("color_code_to_ohms values", _test_b1, 1),
    ("color_code_to_ohms tolerance & 3-band", _test_b2, 1),
    ("color_code_to_ohms errors", _test_b3, 1),
    ("nearest_e12", _test_b4, 1),
    ("validate_netlist valid circuit", _test_b5, 1),
    ("validate_netlist finds every problem", _test_b6, 1),
])

---
## Part C — Numerical methods with NumPy  *(7 pts)*

**Nodal analysis.** For a circuit with nodes 1…N (node 0 = ground) the conductance matrix **G** (N × N) is built by *stamping* every resistor: a resistor `R` between nodes *a* and *b* adds `g = 1/R` to `G[a-1, a-1]` and `G[b-1, b-1]`, and subtracts `g` from `G[a-1, b-1]` and `G[b-1, a-1]`. Terms that involve node 0 are simply skipped.

**RC circuit.** A capacitor charging through `R` from `v0` towards `V_s`: `v(t) = V_s + (v0 − V_s)·e^(−t/RC)`.

| Function | Specification |
|---|---|
| `solve_linear_system(A, b)` | Solve `A x = b` with `np.linalg.solve`. Raise `ValueError` if `A` is not square or sizes mismatch, and convert `np.linalg.LinAlgError` (singular matrix — e.g. a floating node) into a `ValueError`. |
| `stamp_resistor(G, node_a, node_b, ohms)` | Add the resistor's stamp **in place** to the NumPy matrix `G` and return `G`. |
| `rc_step_response(t, r, c, v_source, v0=0.0)` | Capacitor voltage for an array of times `t` (vectorized). |
| `time_to_fraction(t, v, v_final, fraction=0.632)` | First time at which `v ≥ fraction × v_final` (use a boolean mask and `np.argmax`). Return `np.nan` if it never happens. |
| `monte_carlo_divider(v_in, r_top, r_bottom, tol_pct, n=10_000, seed=0)` | Draw both resistors uniformly within ±`tol_pct` % using `np.random.default_rng(seed)` and return the array of `n` output voltages. |

### 📘 Concept review for Part C — NumPy

NumPy works on **whole arrays at once** ("vectorization"): `a * 2` doubles every element without a loop. This is shorter *and* about 100× faster than a Python loop.

| Task | NumPy code |
|---|---|
| Make an array (accepts lists, Series…) | `x = np.asarray(values, dtype=float)` |
| Element-wise maths | `a + b`, `a * 2`, `a ** 2`, `np.sqrt(a)`, `np.exp(a)`, `np.abs(a)` |
| Statistics | `x.sum()`, `x.mean()`, `x.std()`, `x.min()`, `x.max()`, `np.median(x)` |
| Position of the max / sorting order | `np.argmax(x)`, `np.argsort(x)` (ascending — reverse with `[::-1]`) |
| **Boolean mask** (filter) | `x[x > 0]` keeps the positive values; `(x > 0) & (x < 5)` combines conditions |
| Differences between neighbours | `np.diff(x)` → length `len(x) − 1` |
| Moving average | `np.convolve(x, np.ones(n) / n, mode="valid")` |
| Limit to a range | `np.clip(x, 0, None)` (negative values become 0) |
| Straight-line / polynomial fit | `slope, intercept = np.polyfit(x, y, 1)`; evaluate with `np.polyval(coeffs, x)` |
| Same-shape check | `a.shape == b.shape` |

**Tip:** if a test says *"must return a NumPy array"*, wrap your result: `return np.asarray(result)`. Convert NumPy numbers to plain Python numbers with `float(...)` when the task asks for a float.

In [ ]:
# EXAMPLES (not part of the project)
x = np.asarray([3.0, -1.0, 4.0, -1.5, 5.0])
print("doubled:          ", x * 2)
print("positives only:   ", x[x > 0])
print("sum of negatives: ", x[x < 0].sum())
print("index of max:     ", np.argmax(x))
print("largest first:    ", np.argsort(x)[::-1])
print("neighbour diffs:  ", np.diff(x))
print("3-point average:  ", np.convolve(x, np.ones(3) / 3, mode="valid"))
print("clipped at 0:     ", np.clip(x, 0, None))
slope, intercept = np.polyfit([0, 1, 2, 3], [1, 3, 5, 7], 1)
print(f"line fit: y = {slope:.1f} x + {intercept:.1f}")

In [ ]:
def solve_linear_system(A, b):
    """Solve A x = b and return x as a NumPy array."""
    # HINT ── step by step:
    #   1. A = np.asarray(A, dtype=float); b = np.asarray(b, dtype=float)
    #   2. if A.ndim != 2 or A.shape[0] != A.shape[1] or b.shape[0] != A.shape[0]: raise ValueError(...)
    #   3. try: return np.linalg.solve(A, b)
    #      except np.linalg.LinAlgError: raise ValueError("singular matrix ...")
    # YOUR CODE HERE
    raise NotImplementedError()


def stamp_resistor(G, node_a, node_b, ohms):
    """Add the conductance stamp of a resistor between node_a and node_b to G (in place)."""
    # HINT ── see the background section (5). Node k is row/column k - 1:
    #   g = 1.0 / ohms
    #   if node_a != 0: G[node_a - 1, node_a - 1] += g
    #   if node_b != 0: G[node_b - 1, node_b - 1] += g
    #   if node_a != 0 and node_b != 0: G[node_a - 1, node_b - 1] -= g and G[node_b - 1, node_a - 1] -= g
    #   return G
    # YOUR CODE HERE
    raise NotImplementedError()


def rc_step_response(t, r, c, v_source, v0=0.0):
    """Return the capacitor voltage of an RC circuit at the times in t."""
    # HINT ── vectorized formula:
    #   t = np.asarray(t, dtype=float)
    #   return v_source + (v0 - v_source) * np.exp(-t / (r * c))
    # YOUR CODE HERE
    raise NotImplementedError()


def time_to_fraction(t, v, v_final, fraction=0.632):
    """Return the first time at which v reaches fraction * v_final (np.nan if never)."""
    # HINT ── boolean mask + argmax (argmax of a True/False array = index of the first True):
    #   mask = np.asarray(v, dtype=float) >= fraction * v_final
    #   if not mask.any(): return np.nan
    #   return float(np.asarray(t, dtype=float)[np.argmax(mask)])
    # YOUR CODE HERE
    raise NotImplementedError()


def monte_carlo_divider(v_in, r_top, r_bottom, tol_pct, n=10_000, seed=0):
    """Return n Monte Carlo samples of a divider's output voltage with toleranced resistors."""
    # HINT ── step by step:
    #   1. rng = np.random.default_rng(seed); tol = tol_pct / 100
    #   2. rt = r_top * (1 + rng.uniform(-tol, tol, n))       (n random top resistors)
    #   3. rb = r_bottom * (1 + rng.uniform(-tol, tol, n))    (n random bottom resistors)
    #   4. return v_in * rb / (rt + rb)                        (n output voltages at once)
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
def _test_c1():
    x = solve_linear_system([[2, 1], [1, 3]], [3, 5])
    assert_close(x, [0.8, 1.4])

def _test_c2():
    assert_raises(ValueError, solve_linear_system, [[1, 2], [2, 4]], [1, 2])
    assert_raises(ValueError, solve_linear_system, [[1, 2, 3], [4, 5, 6]], [1, 2])

def _test_c3():
    G = np.zeros((2, 2))
    out = stamp_resistor(G, 1, 2, 100)
    assert out is G, "must modify G in place and return it"
    stamp_resistor(G, 2, 0, 50)
    assert_close(G, [[0.01, -0.01], [-0.01, 0.03]])

def _test_c4():
    tau = 1e3 * 1e-6
    t = np.array([0, tau, 5 * tau])
    v = rc_step_response(t, 1e3, 1e-6, 5.0)
    assert_close(v, [0.0, 5 * (1 - np.exp(-1)), 5 * (1 - np.exp(-5))])
    assert_close(rc_step_response([0.0], 1e3, 1e-6, 0.0, v0=3.3), [3.3])

def _test_c5():
    t = np.linspace(0, 0.01, 10_001)
    v = rc_step_response(t, 1e3, 1e-6, 5.0)
    assert_close(time_to_fraction(t, v, 5.0), 1e-3, tol=2e-6)
    assert np.isnan(time_to_fraction(t, v, 5.0, fraction=1.5))

def _test_c6():
    s = monte_carlo_divider(12, 10e3, 4.7e3, 5, n=20_000, seed=1)
    assert isinstance(s, np.ndarray) and s.shape == (20_000,)
    lo = 12 * 4.7e3 * 0.95 / (10e3 * 1.05 + 4.7e3 * 0.95)
    hi = 12 * 4.7e3 * 1.05 / (10e3 * 0.95 + 4.7e3 * 1.05)
    assert s.min() >= lo and s.max() <= hi
    assert abs(s.mean() - 12 * 4.7 / 14.7) < 0.02

def _test_c7():
    a = monte_carlo_divider(5, 1e3, 1e3, 1, n=100, seed=42)
    b = monte_carlo_divider(5, 1e3, 1e3, 1, n=100, seed=42)
    assert np.array_equal(a, b), "same seed must give the same samples"
    assert_close(monte_carlo_divider(5, 1e3, 1e3, 0, n=10), np.full(10, 2.5))

run_tests("C", [
    ("solve_linear_system", _test_c1, 1),
    ("solve_linear_system singular / bad shape", _test_c2, 1),
    ("stamp_resistor in place", _test_c3, 1),
    ("rc_step_response", _test_c4, 1),
    ("time_to_fraction (time constant)", _test_c5, 1),
    ("monte_carlo_divider bounds and mean", _test_c6, 1),
    ("monte_carlo_divider reproducibility", _test_c7, 1),
])

---
## Part D — Data pipeline with Pandas  *(7 pts)*

| Function | Specification |
|---|---|
| `load_netlists(path)` | `pd.read_csv(path, dtype=str)` — keep everything as text for now. |
| `clean_netlist(df)` | Return a new DataFrame: (1) strip spaces from every text cell; (2) drop comment rows (`circuit_id` starts with `#`); (3) upper-case `type`; (4) drop exact duplicate rows; (5) convert `node_pos`/`node_neg` to `int`; (6) convert `value` with `parse_si_value` — **drop rows that cannot be parsed**; (7) `tolerance_pct` to float, missing → 5.0 for resistors and 0.0 for sources; (8) reset the index. |
| `netlist_summary(clean)` | One row per `circuit_id` with columns `n_components, n_resistors, n_sources, n_nodes` (`n_nodes` = largest node number). |
| `decode_inventory(inv)` | Add columns `ohms`, `tolerance_pct` (from `color_code_to_ohms`) and `stock_value_usd` (= quantity × unit cost). |
| `check_inventory(clean, inventory)` | For every **resistor** in the netlists: `circuit_id, name, value, quantity, in_stock`. Sum the quantities of inventory parts with the same resistance first, match on resistance (round both to 6 decimals), missing → quantity 0. `in_stock` is `True` when quantity > 0. |
| `monte_carlo_table(samples, spec_low, spec_high)` | `samples` is a dict `label → NumPy array`. Return a DataFrame (index = label) with columns `mean, std, min, max, yield_pct` where `yield_pct` = % of samples inside `[spec_low, spec_high]`. |

### 📘 Concept review for Part D — Pandas

A **DataFrame** is a table: columns have names, rows have an index. Cleaning real data almost always follows the same recipe:

1. **Copy** the input so you do not change it: `out = df.copy()` (or start from `df.drop_duplicates()`, which already returns a new table)
2. **Remove duplicates**: `out = out.drop_duplicates()`
3. **Fix types**: `out["col"] = pd.to_numeric(out["col"], errors="coerce")` → text such as `"ERR"` becomes `NaN`
4. **Mark impossible values as missing**: `out.loc[out["col"] < 0, "col"] = np.nan`
5. **Sort**: `out = out.sort_values(["id", "time"]).reset_index(drop=True)`
6. **Fill the gaps inside each group**: `out["col"] = out.groupby("id")["col"].transform(lambda s: s.interpolate().bfill().ffill())`

| Task | Pandas code |
|---|---|
| Read a CSV / parse dates | `pd.read_csv(path)`, `pd.to_datetime(df["timestamp"])` |
| Select rows | `df[df["id"] == "A"]`, `df.loc[mask, "col"]` |
| New column | `df["c"] = df["a"] * df["b"]` or `df.assign(c=...)` |
| Date parts | `df["timestamp"].dt.hour`, `.dt.date` |
| Group and summarise | `df.groupby("id").agg(total=("x", "sum"), peak=("x", "max")).reset_index()` |
| Join two tables | `left.merge(right, on="id", how="left")` |
| Wide table | `df.pivot_table(index="hour", columns="id", values="x", aggfunc="mean")` |
| Apply a Python function to each value | `df["x"].map(my_function)` |

`interpolate()` fills a gap with the straight line between its neighbours (1, NaN, 3 → 1, 2, 3); `bfill()`/`ffill()` copy the next/previous value into gaps at the very start or end.

In [ ]:
# EXAMPLE (not part of the project) — the cleaning recipe on a tiny table
raw = pd.DataFrame({"id": ["A", "A", "A", "B", "B", "B"],
                    "t": [2, 0, 1, 0, 1, 1],
                    "x": ["3.0", "1.0", "ERR", "-5", "4.0", "4.0"]})
clean = raw.drop_duplicates().copy()                            # removes the repeated B row
clean["x"] = pd.to_numeric(clean["x"], errors="coerce")         # "ERR" -> NaN
clean.loc[clean["x"] < 0, "x"] = np.nan                         # impossible value -> NaN
clean = clean.sort_values(["id", "t"]).reset_index(drop=True)
clean["x"] = clean.groupby("id")["x"].transform(lambda s: s.interpolate().bfill().ffill())
print(clean)
print(clean.groupby("id").agg(total=("x", "sum"), peak=("x", "max")).reset_index())

In [ ]:
def load_netlists(path):
    """Read the netlist CSV with every column as text."""
    # HINT ── one line: return pd.read_csv(path, dtype=str)
    # YOUR CODE HERE
    raise NotImplementedError()


def _try_parse(text):
    """Return parse_si_value(text) or NaN if the text is not a valid value."""
    try:
        return parse_si_value(text)
    except (ValueError, TypeError):
        return np.nan


def clean_netlist(df):
    """Return a cleaned copy of the netlist table (see the table above)."""
    # HINT ── step by step (keep the order):
    #   1. out = df.copy(); for col in out.columns: out[col] = out[col].map(lambda x: x.strip() if isinstance(x, str) else x)
    #   2. out = out[~out["circuit_id"].str.startswith("#")].copy()          (~ means NOT)
    #   3. out["type"] = out["type"].str.upper();   out = out.drop_duplicates()
    #   4. out["node_pos"] = pd.to_numeric(out["node_pos"]).astype(int)   (same for node_neg)
    #   5. out["value"] = out["value"].map(_try_parse).astype(float); out = out.dropna(subset=["value"])
    #   6. tol = pd.to_numeric(out["tolerance_pct"], errors="coerce")
    #      default = np.where(out["type"] == "R", 5.0, 0.0)
    #      out["tolerance_pct"] = tol.fillna(pd.Series(default, index=out.index)).astype(float)
    #   7. return out.reset_index(drop=True)
    # YOUR CODE HERE
    raise NotImplementedError()


def netlist_summary(clean):
    """Return the number of components, resistors, sources and nodes of each circuit."""
    # HINT ── add helper columns, then group:
    #   tmp = clean.assign(is_r=clean["type"].eq("R"), is_src=clean["type"].isin(["V", "I"]),
    #                      max_node=clean[["node_pos", "node_neg"]].max(axis=1))
    #   return tmp.groupby("circuit_id").agg(n_components=("name", "count"), n_resistors=("is_r", "sum"),
    #                                        n_sources=("is_src", "sum"), n_nodes=("max_node", "max"))
    # YOUR CODE HERE
    raise NotImplementedError()


def decode_inventory(inv):
    """Return the inventory with decoded resistance, tolerance and stock value."""
    # HINT ── step by step:
    #   1. out = inv.copy(); decoded = out["bands"].map(color_code_to_ohms)    (a Series of (ohms, tol) tuples)
    #   2. out["ohms"] = [d[0] for d in decoded]; out["tolerance_pct"] = [d[1] for d in decoded]
    #   3. out["stock_value_usd"] = out["quantity"] * out["unit_cost_usd"]; return out
    # YOUR CODE HERE
    raise NotImplementedError()


def check_inventory(clean, inventory):
    """Return, for every resistor in the netlists, how many matching parts are in stock."""
    # HINT ── step by step:
    #   1. stock = inventory.assign(key=inventory["ohms"].round(6)).groupby("key", as_index=False)["quantity"].sum()
    #   2. res = clean[clean["type"] == "R"].assign(key=lambda d: d["value"].round(6))
    #   3. merged = res.merge(stock, on="key", how="left")
    #   4. merged["quantity"] = merged["quantity"].fillna(0).astype(int); merged["in_stock"] = merged["quantity"] > 0
    #   5. return merged[["circuit_id", "name", "value", "quantity", "in_stock"]]
    # YOUR CODE HERE
    raise NotImplementedError()


def monte_carlo_table(samples, spec_low, spec_high):
    """Summarize Monte Carlo samples (dict label -> array) and the manufacturing yield."""
    # HINT ── build a dict of dicts, then a DataFrame:
    #   rows = {}
    #   for label, s in samples.items():
    #       s = np.asarray(s, dtype=float)
    #       rows[label] = {"mean": s.mean(), "std": s.std(), "min": s.min(), "max": s.max(),
    #                      "yield_pct": 100.0 * np.mean((s >= spec_low) & (s <= spec_high))}
    #   return pd.DataFrame.from_dict(rows, orient="index")
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
_SMALL_NET = pd.DataFrame({
    "circuit_id": ["T1", " T1", "# comment", "T1", "T1", "T1"],
    "name": ["V1", "R1 ", "", "R2", "R2", "R3"],
    "type": ["v", " R", "", "r", "r", "R"],
    "node_pos": ["1", "1", "", "2.0", "2.0", "2"],
    "node_neg": ["0", "2", "", "0", "0", "0"],
    "value": ["5V", "1k", "", "2.2 k", "2.2 k", "bad"],
    "tolerance_pct": [np.nan, "1", np.nan, np.nan, np.nan, np.nan],
})

def _test_d1():
    raw = load_netlists(NETLIST_CSV)
    assert len(raw) == 29 and list(raw.columns)[:3] == ["circuit_id", "name", "type"]

def _test_d2():
    before = _SMALL_NET.copy()
    c = clean_netlist(_SMALL_NET)
    pd.testing.assert_frame_equal(_SMALL_NET, before)
    assert list(c["name"]) == ["V1", "R1", "R2"]
    assert list(c["type"]) == ["V", "R", "R"]
    assert c["node_pos"].tolist() == [1, 1, 2] and pd.api.types.is_integer_dtype(c["node_pos"])
    assert_close(c["value"].tolist(), [5.0, 1000.0, 2200.0])
    assert c["tolerance_pct"].tolist() == [0.0, 1.0, 5.0]

def _test_d3():
    c = clean_netlist(load_netlists(NETLIST_CSV))
    assert len(c) == 26, f"expected 26 components, got {len(c)}"
    assert set(c["type"]) == {"R", "V", "I"}
    assert "RX" not in set(c["name"])
    assert c.index.equals(pd.RangeIndex(len(c)))

def _test_d4():
    s = netlist_summary(clean_netlist(load_netlists(NETLIST_CSV)))
    assert s.loc["C2", "n_resistors"] == 5 and s.loc["C2", "n_nodes"] == 3
    assert s.loc["C5", "n_sources"] == 2 and s.loc["C3", "n_components"] == 8
    assert int(s["n_components"].sum()) == 26

def _test_d5():
    inv = decode_inventory(pd.read_csv(INVENTORY_CSV))
    assert_close(inv.loc[inv["part_id"] == "P002", "ohms"].iloc[0], 4700)
    assert inv.loc[inv["part_id"] == "P010", "tolerance_pct"].iloc[0] == 20.0
    assert_close(inv["stock_value_usd"].sum(), 17.55)

def _test_d6():
    c = clean_netlist(load_netlists(NETLIST_CSV))
    chk = check_inventory(c, decode_inventory(pd.read_csv(INVENTORY_CSV)))
    assert len(chk) == (c["type"] == "R").sum()
    one_k = chk[(chk["circuit_id"] == "C2") & (chk["name"] == "R1")].iloc[0]
    assert one_k["quantity"] == 230, "two 1 kΩ parts must be summed (30 + 200)"
    two_k = chk[(chk["circuit_id"] == "C3") & (chk["name"] == "R2")].iloc[0]
    assert two_k["quantity"] == 0 and not two_k["in_stock"]

def _test_d7():
    t = monte_carlo_table({"a": np.array([1.0, 2.0, 3.0, 4.0]), "b": np.array([2.0, 2.0])}, 1.5, 3.5)
    assert list(t.index) == ["a", "b"]
    assert_close(t.loc["a", "yield_pct"], 50.0); assert_close(t.loc["b", "yield_pct"], 100.0)
    assert_close(t.loc["a", "mean"], 2.5); assert_close(t.loc["a", "max"], 4.0)

run_tests("D", [
    ("load_netlists", _test_d1, 1),
    ("clean_netlist on a small hand-made DataFrame", _test_d2, 1),
    ("clean_netlist on the full dataset", _test_d3, 1),
    ("netlist_summary", _test_d4, 1),
    ("decode_inventory", _test_d5, 1),
    ("check_inventory merge", _test_d6, 1),
    ("monte_carlo_table", _test_d7, 1),
])

---
## Part E — Object-oriented circuit model  *(7 pts)*

```
Component (ABC)                                  Circuit
 ├── Resistor                                     ◆── holds many Component objects (composition)
 ├── VoltageSource   (PROVIDED — worked example)  ── solve(), branch_report(), from_netlist()
 └── CurrentSource
```

**How the pieces fit together:** `Circuit.solve()` creates an all-zero matrix `A` and vector `z`, asks **every component to stamp itself** (polymorphism — `comp.stamp(A, z, n)` does something different for each class), then calls your Part C `solve_linear_system(A, z)`. You only need to write the Resistor and CurrentSource stamps; the voltage-source stamp is provided.

**Modified Nodal Analysis.** With N nodes and M voltage sources, solve `A x = z`, where `A` is (N+M) × (N+M). The first N unknowns are node voltages, the last M are the currents through the voltage sources. Each component *stamps* itself into `A` and `z`:

| Component | Stamp (node k ↦ row/column k−1, skip ground) |
|---|---|
| Resistor between a, b | `stamp_resistor` (Part C) on the top-left N × N block of `A` |
| CurrentSource pushing `I` **into** node `node_pos` (out of `node_neg`) | `z[pos−1] += I`, `z[neg−1] −= I` |
| VoltageSource #k: `V(node_pos) − V(node_neg) = V` | row/col `j = N + k`: `A[pos−1, j] += 1`, `A[j, pos−1] += 1`, `A[neg−1, j] −= 1`, `A[j, neg−1] −= 1`, `z[j] = V` |

**Sign convention for every component:** `current(circuit)` is the current flowing **through the component from `node_pos` to `node_neg`**, and `power(circuit) = voltage_drop × current` is the power **absorbed** (negative = delivering). With this convention the powers of all components add up to zero (Tellegen's theorem) — a great self-check!

**`Component(ABC)`** — `__init__(self, name, node_pos, node_neg, value)`; raise `ValueError` if the nodes are equal or negative.
* abstract `stamp(self, A, z, n_nodes)` and abstract `current(self, circuit)`
* concrete `voltage_drop(self, circuit)` = `circuit.voltage(node_pos) − circuit.voltage(node_neg)`; concrete `power(self, circuit)`
* `__repr__` → e.g. `"Resistor('R1', 1, 2, 1000.0)"`

| Subclass | Notes |
|---|---|
| `Resistor(name, a, b, ohms, tolerance_pct=5.0)` | `ValueError` if `ohms <= 0`; current = drop / R |
| `VoltageSource(name, pos, neg, volts)` | **PROVIDED** — read it as a worked example of a subclass. `branch_index` is set by `Circuit.solve`; current = the solved MNA current |
| `CurrentSource(name, pos, neg, amps)` | current from pos to neg through the source = `−amps` |

**`Circuit(name)`**
* `add(component)` → `TypeError` if not a `Component`, `ValueError` for a duplicate name; **returns `self`** so calls can be chained.
* `__len__`, `__contains__(name)`, `__getitem__(name)`; property `n_nodes` (largest node number)
* `solve()` → builds `A`, `z`, uses `solve_linear_system`, stores and returns `node_voltages` (dict including `0: 0.0`); also stores `source_currents` (dict name → current)
* `voltage(node)` → `RuntimeError` if the circuit has not been solved
* `branch_report()` → DataFrame with columns `name, type, voltage_drop, current_a, power_w` (`type` = class name)
* `@classmethod from_netlist(cls, clean_df, circuit_id)` → build a circuit from cleaned netlist rows (a dict `{"R": Resistor, "V": VoltageSource, "I": CurrentSource}` acts as a *factory*)

### 📘 Concept review for Part E — Object-Oriented Programming

| Idea | Meaning | Python |
|---|---|---|
| **Class / object** | a blueprint / one thing built from it | `class Motor:` → `m = Motor("M1", 2.0)` |
| **`__init__` and `self`** | store the data of each object | `self.power_kw = power_kw` |
| **Method** | a function that belongs to the object | `def energy(self, hours): return self.power_kw * hours` |
| **Inheritance** | a subclass *is a* special kind of the base class | `class ServoMotor(Motor):` and call `super().__init__(...)` |
| **Abstract base class** | a base class that only defines the interface; it cannot be created | `class Shape(ABC):` + `@abstractmethod` |
| **Polymorphism** | the same method call does different things for different classes | `for s in shapes: s.area()` |
| **Composition** | an object *has* other objects | `self.motors = []` + `add_motor(m)` |
| **Property** | a method used like an attribute | `@property def area(self): ...` → `s.area` (no parentheses) |
| **Dunder methods** | let your objects work with built-ins | `__repr__` (printing), `__len__` (`len()`), `__lt__` (`<`, `sorted()`, `min()`), `__iter__` (`for`), `__getitem__` (`obj[key]`) |
| **Class method** | an alternative constructor | `@classmethod def from_dataframe(cls, df): ...` |
| **Type check** | is this object of the right class? | `isinstance(x, Motor)` → raise `TypeError` if not |

In [ ]:
# EXAMPLE (not part of the project) — every OOP idea used in Part E, in 40 lines
class Shape(ABC):
    """Abstract base class: every shape has a name and an area."""

    def __init__(self, name):
        self.name = name

    @abstractmethod
    def area(self):
        """Return the area (each subclass must implement this)."""

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, area={self.area():.2f})"

    def __lt__(self, other):                  # lets sorted()/min()/max() compare shapes
        return self.area() < other.area()


class Rectangle(Shape):
    def __init__(self, name, w, h):
        super().__init__(name)                # reuse the base-class constructor
        self.w, self.h = w, h

    def area(self):
        return self.w * self.h


class Circle(Shape):
    def __init__(self, name, r):
        if r <= 0:
            raise ValueError("radius must be positive")
        super().__init__(name)
        self.r = r

    def area(self):
        return math.pi * self.r ** 2


class Drawing:
    """Composition: a drawing HAS many shapes."""

    def __init__(self):
        self.shapes = []

    def add(self, shape):
        if not isinstance(shape, Shape):
            raise TypeError("only Shape objects can be added")
        self.shapes.append(shape)
        return self                           # returning self allows chaining: d.add(a).add(b)

    def __len__(self):
        return len(self.shapes)

    def total_area(self):
        return sum(s.area() for s in self.shapes)   # polymorphism: area() differs per class


d = Drawing().add(Rectangle("door", 1, 2)).add(Circle("window", 0.5))
print(len(d), "shapes, total area", round(d.total_area(), 3))
print(sorted(d.shapes), "| largest:", max(d.shapes).name)
try:
    Shape("x")
except TypeError as err:
    print("Abstract class:", err)

In [ ]:
class Component(ABC):
    """Abstract two-terminal circuit element."""

    def __init__(self, name, node_pos, node_neg, value):
        # HINT ── validate, then store:
        #   if node_pos == node_neg: raise ValueError(...);  if node_pos < 0 or node_neg < 0: raise ValueError(...)
        #   self.name = name; self.node_pos = int(node_pos); self.node_neg = int(node_neg); self.value = float(value)
        # YOUR CODE HERE
        raise NotImplementedError()

    @abstractmethod
    def stamp(self, A, z, n_nodes):
        """Add this component's contribution to the MNA matrix A and vector z."""

    @abstractmethod
    def current(self, circuit):
        """Return the current through the component from node_pos to node_neg (A)."""

    def voltage_drop(self, circuit):
        """Return V(node_pos) - V(node_neg) in volts."""
        # HINT ── return circuit.voltage(self.node_pos) - circuit.voltage(self.node_neg)
        # YOUR CODE HERE
        raise NotImplementedError()

    def power(self, circuit):
        """Return the power absorbed by the component (W); negative means delivering."""
        # HINT ── return self.voltage_drop(circuit) * self.current(circuit)
        #   (self.current is the SUBCLASS method — polymorphism)
        # YOUR CODE HERE
        raise NotImplementedError()

    def __repr__(self):
        # HINT ── return f"{type(self).__name__}({self.name!r}, {self.node_pos}, {self.node_neg}, {self.value})"
        # YOUR CODE HERE
        raise NotImplementedError()


class Resistor(Component):
    """Linear resistor."""

    def __init__(self, name, node_pos, node_neg, ohms, tolerance_pct=5.0):
        # HINT ── if ohms <= 0: raise ValueError(...)
        #   super().__init__(name, node_pos, node_neg, ohms); self.tolerance_pct = tolerance_pct
        # YOUR CODE HERE
        raise NotImplementedError()

    def stamp(self, A, z, n_nodes):
        # HINT ── reuse Part C: stamp_resistor(A, self.node_pos, self.node_neg, self.value)
        # YOUR CODE HERE
        raise NotImplementedError()

    def current(self, circuit):
        # HINT ── Ohm's law: return self.voltage_drop(circuit) / self.value
        # YOUR CODE HERE
        raise NotImplementedError()


class VoltageSource(Component):
    """Ideal independent DC voltage source.

    PROVIDED — a complete worked example of a Component subclass. Read it; do not change it.
    """

    def __init__(self, name, node_pos, node_neg, volts):
        super().__init__(name, node_pos, node_neg, volts)
        self.branch_index = None

    def stamp(self, A, z, n_nodes):
        j = n_nodes + self.branch_index
        if self.node_pos != 0:
            A[self.node_pos - 1, j] += 1
            A[j, self.node_pos - 1] += 1
        if self.node_neg != 0:
            A[self.node_neg - 1, j] -= 1
            A[j, self.node_neg - 1] -= 1
        z[j] = self.value

    def current(self, circuit):
        return circuit.source_currents[self.name]


class CurrentSource(Component):
    """Ideal independent DC current source pushing `value` amps into node_pos."""

    def stamp(self, A, z, n_nodes):
        # HINT ── the source pushes self.value amps INTO node_pos and OUT of node_neg (skip ground):
        #   if self.node_pos != 0: z[self.node_pos - 1] += self.value
        #   if self.node_neg != 0: z[self.node_neg - 1] -= self.value
        # YOUR CODE HERE
        raise NotImplementedError()

    def current(self, circuit):
        # HINT ── current from node_pos to node_neg THROUGH the source is opposite to the pushed current:
        #   return -self.value
        # YOUR CODE HERE
        raise NotImplementedError()


class Circuit:
    """A collection of components that can be solved with Modified Nodal Analysis."""

    def __init__(self, name):
        # HINT ── self.name = name; self.components = {}   (dict: name -> Component)
        #   self.node_voltages = None; self.source_currents = None
        # YOUR CODE HERE
        raise NotImplementedError()

    def add(self, component):
        # HINT ── if not isinstance(component, Component): raise TypeError(...)
        #   if component.name in self.components: raise ValueError(...)
        #   self.components[component.name] = component; return self
        # YOUR CODE HERE
        raise NotImplementedError()

    def __len__(self):
        # HINT ── return len(self.components)
        # YOUR CODE HERE
        raise NotImplementedError()

    def __contains__(self, name):
        # HINT ── return name in self.components
        # YOUR CODE HERE
        raise NotImplementedError()

    def __getitem__(self, name):
        # HINT ── return self.components[name]
        # YOUR CODE HERE
        raise NotImplementedError()

    @property
    def n_nodes(self):
        # HINT ── the largest node number used by any component:
        #   return max(max(c.node_pos, c.node_neg) for c in self.components.values())
        # YOUR CODE HERE
        raise NotImplementedError()

    def solve(self):
        """Solve the circuit and return a dict node -> voltage (including ground)."""
        # HINT ── step by step:
        #   1. n = self.n_nodes
        #   2. sources = [c for c in self.components.values() if isinstance(c, VoltageSource)]
        #      for k, src in enumerate(sources): src.branch_index = k
        #   3. size = n + len(sources); A = np.zeros((size, size)); z = np.zeros(size)
        #   4. for comp in self.components.values(): comp.stamp(A, z, n)        <- polymorphism
        #   5. x = solve_linear_system(A, z)
        #   6. self.node_voltages = {0: 0.0}, then for k in range(n): self.node_voltages[k + 1] = float(x[k])
        #   7. self.source_currents = {src.name: float(x[n + src.branch_index]) for src in sources}
        #   8. return self.node_voltages
        # YOUR CODE HERE
        raise NotImplementedError()

    def voltage(self, node):
        # HINT ── if self.node_voltages is None: raise RuntimeError("call solve() first")
        #   return self.node_voltages[node]
        # YOUR CODE HERE
        raise NotImplementedError()

    def branch_report(self):
        # HINT ── one dict per component, then a DataFrame:
        #   rows = [{"name": c.name, "type": type(c).__name__, "voltage_drop": c.voltage_drop(self),
        #            "current_a": c.current(self), "power_w": c.power(self)} for c in self.components.values()]
        #   return pd.DataFrame(rows)
        # YOUR CODE HERE
        raise NotImplementedError()

    @classmethod
    def from_netlist(cls, clean_df, circuit_id):
        # HINT ── a "factory": choose the class from the type letter
        #   1. factory = {"R": Resistor, "V": VoltageSource, "I": CurrentSource}; circuit = cls(circuit_id)
        #   2. rows = clean_df[clean_df["circuit_id"] == circuit_id]; if rows.empty: raise ValueError(...)
        #   3. for _, r in rows.iterrows():
        #          if r["type"] == "R": comp = Resistor(r["name"], r["node_pos"], r["node_neg"], r["value"], r["tolerance_pct"])
        #          else: comp = factory[r["type"]](r["name"], r["node_pos"], r["node_neg"], r["value"])
        #          circuit.add(comp)
        #   4. return circuit
        # YOUR CODE HERE
        raise NotImplementedError()


In [ ]:
def _divider():
    return (Circuit("div").add(VoltageSource("V1", 1, 0, 12.0))
            .add(Resistor("R1", 1, 2, 10e3)).add(Resistor("R2", 2, 0, 4.7e3)))

def _test_e1():
    assert_raises(TypeError, Component, "X", 1, 0, 1.0)
    assert_raises(ValueError, Resistor, "R1", 1, 1, 100)
    assert_raises(ValueError, Resistor, "R1", 1, 0, -100)
    assert repr(Resistor("R1", 1, 2, 1000)) == "Resistor('R1', 1, 2, 1000.0)"

def _test_e2():
    c = _divider()
    assert len(c) == 3 and "R2" in c and "R9" not in c and c.n_nodes == 2
    assert isinstance(c["V1"], VoltageSource)
    assert_raises(ValueError, c.add, Resistor("R1", 1, 0, 5))
    assert_raises(TypeError, c.add, "R3")
    assert_raises(RuntimeError, c.voltage, 2)

def _test_e3():
    v = _divider().solve()
    assert_close(v[0], 0.0); assert_close(v[1], 12.0); assert_close(v[2], 12 * 4.7 / 14.7)

def _test_e4():
    c = (Circuit("isrc").add(CurrentSource("I1", 1, 0, 2e-3)).add(Resistor("R1", 1, 0, 1e3))
         .add(Resistor("R2", 1, 2, 3.3e3)).add(Resistor("R3", 2, 0, 6.8e3)))
    v = c.solve()
    v1 = 2e-3 * parallel_resistance([1e3, 10.1e3])
    assert_close(v[1], v1); assert_close(v[2], v1 * 6.8 / 10.1)
    assert_close(c["I1"].current(c), -2e-3)

def _test_e5():
    c = _divider(); c.solve()
    rep = c.branch_report()
    assert list(rep.columns) == ["name", "type", "voltage_drop", "current_a", "power_w"]
    i = 12 / 14.7e3
    assert_close(rep.set_index("name").loc["R1", "current_a"], i)
    assert_close(rep.set_index("name").loc["V1", "power_w"], -12 * i)
    assert_close(rep["power_w"].sum(), 0.0, tol=1e-12)    # Tellegen's theorem

def _test_e6():
    c = (Circuit("two").add(VoltageSource("V1", 1, 0, 10.0)).add(VoltageSource("V2", 3, 0, 5.0))
         .add(Resistor("R1", 1, 2, 1e3)).add(Resistor("R2", 2, 3, 2e3)).add(Resistor("R3", 2, 0, 3.3e3)))
    v = c.solve()
    v2 = (10 / 1e3 + 5 / 2e3) / (1 / 1e3 + 1 / 2e3 + 1 / 3.3e3)
    assert_close(v[2], v2); assert_close(c.voltage(3), 5.0)
    assert sorted(c.source_currents) == ["V1", "V2"]
    assert_close(c["V1"].current(c), -(10 - v2) / 1e3)

def _test_e7():
    clean = clean_netlist(load_netlists(NETLIST_CSV))
    c5 = Circuit.from_netlist(clean, "C5")
    assert len(c5) == 5 and isinstance(c5["V2"], VoltageSource) and isinstance(c5["R3"], Resistor)
    v2 = (10 / 1e3 + 5 / 2e3) / (1 / 1e3 + 1 / 2e3 + 1 / 3.3e3)
    assert_close(c5.solve()[2], v2)
    assert c5["R1"].tolerance_pct == 5.0
    assert_raises(ValueError, Circuit.from_netlist, clean, "C99")

run_tests("E", [
    ("abstract base class, validation, __repr__", _test_e1, 1),
    ("Circuit container: add, len, in, [], n_nodes", _test_e2, 1),
    ("solve a voltage divider", _test_e3, 1),
    ("solve a current-source circuit", _test_e4, 1),
    ("branch_report and energy conservation", _test_e5, 1),
    ("two voltage sources, source currents", _test_e6, 1),
    ("from_netlist factory (two sources)", _test_e7, 1),
])

---
## Part F — Integration, report & command-line interface  *(3 pts + used in your video demo)*

| Function | Specification |
|---|---|
| `solve_all_circuits(clean_df)` | Return a dict `circuit_id → Circuit`, each already **solved** (use `Circuit.from_netlist` and `solve`). |
| `circuit_table(circuits)` | DataFrame indexed by `circuit_id` with columns `n_components`, `n_nodes`, `max_voltage` (largest node voltage) and `resistor_power_w` (total power absorbed by all `Resistor` objects). |
| `run_cli(circuits, input_func=input)` | Menu loop: `1` list circuits · `2` branch report (ask for a circuit id, case-insensitive; handle unknown ids) · `3` node voltages of one circuit (same id handling) · `q` quit · anything else prints `"Invalid choice"`. Return the list of menu choices entered. |

### 📘 Concept review for Part F — a menu-driven program you can test

A command-line interface (CLI) is a `while True` loop that shows a menu, reads a choice, uses `if/elif/else` to act on it, and `return`s when the user quits.

**Why `input_func=input`?** A test cannot type on a keyboard. By passing the input function as a parameter, the test can replace the keyboard with a list of pre-written answers (`_scripted([...])`), while you still call `run_cli(...)` normally to type yourself.

Rules the tests check: (1) every menu choice you read is appended to `history` (choices only — not the extra answers such as an id); (2) unknown choices print `Invalid choice` and the loop continues; (3) a wrong id prints a message and **continues** (use `continue`) instead of crashing; (4) `q` returns `history`.

In [ ]:
# EXAMPLE (not part of the project) — a tiny testable menu
def unit_converter_cli(input_func=input):
    history = []
    while True:
        print("1) km -> miles   2) °C -> °F   q) quit")
        choice = input_func("Choice: ").strip().lower()
        history.append(choice)
        if choice == "1":
            try:
                km = float(input_func("km: "))
            except ValueError:
                print("Please type a number")
                continue
            print(f"{km} km = {km * 0.621371:.2f} miles")
        elif choice == "2":
            c = float(input_func("°C: "))
            print(f"{c} °C = {c * 9 / 5 + 32:.1f} °F")
        elif choice == "q":
            return history
        else:
            print("Invalid choice")


answers = iter(["1", "10", "1", "ten", "7", "2", "100", "q"])     # pretend keyboard
print(unit_converter_cli(input_func=lambda prompt="": next(answers)))

In [ ]:
def solve_all_circuits(clean_df):
    """Build and solve every circuit in the cleaned netlist."""
    # HINT ── circuits = {}
    #   for cid in sorted(clean_df["circuit_id"].unique()):
    #       circuit = Circuit.from_netlist(clean_df, cid); circuit.solve(); circuits[cid] = circuit
    #   return circuits
    # YOUR CODE HERE
    raise NotImplementedError()


def circuit_table(circuits):
    """Return one summary row per solved circuit."""
    # HINT ── build one dict per circuit:
    #   rows[cid] = {"n_components": len(c), "n_nodes": c.n_nodes, "max_voltage": max(c.node_voltages.values()),
    #                "resistor_power_w": sum(r.power(c) for r in c.components.values() if isinstance(r, Resistor))}
    #   table = pd.DataFrame.from_dict(rows, orient="index"); table.index.name = "circuit_id"; return table
    # YOUR CODE HERE
    raise NotImplementedError()


def run_cli(circuits, input_func=input):
    """Menu-driven interface to VoltLab. Returns the list of menu choices entered."""
    # HINT ── copy the structure of the concept-review example:
    #   history = []; while True: print menu; choice = input_func("Choice: ").strip().lower(); history.append(choice)
    #   "1": print each circuit id with len(c) and c.n_nodes
    #   "2" or "3": cid = input_func("Circuit id: ").strip().upper(); if cid not in circuits: print message, continue
    #        "2": print(circuits[cid].branch_report())   "3": print every node voltage
    #   "q": return history        anything else: print("Invalid choice")
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
def _scripted(answers):
    it = iter(answers)
    return lambda prompt="": next(it)

def _circuits():
    return solve_all_circuits(clean_netlist(load_netlists(NETLIST_CSV)))

def _test_f1():
    circuits = _circuits()
    assert sorted(circuits) == ["C1", "C2", "C3", "C4", "C5"]
    assert all(c.node_voltages is not None for c in circuits.values())
    c3 = circuits["C3"]
    assert_close([c3.voltage(k) for k in (2, 3, 4)], [4.5, 2.25, 1.125], msg="R-2R ladder halves the voltage")
    for c in circuits.values():
        assert_close(c.branch_report()["power_w"].sum(), 0.0, tol=1e-12)

def _test_f2():
    t = circuit_table(_circuits())
    assert list(t.index) == ["C1", "C2", "C3", "C4", "C5"]
    assert list(t.columns) == ["n_components", "n_nodes", "max_voltage", "resistor_power_w"]
    assert t.loc["C3", "n_components"] == 8 and t.loc["C3", "n_nodes"] == 4
    assert_close(t.loc["C1", "resistor_power_w"], 12 ** 2 / 14.7e3); assert_close(t.loc["C1", "max_voltage"], 12.0)

def _test_f3():
    hist = run_cli(_circuits(), input_func=_scripted(["1", "2", "c2", "2", "C9", "3", "c4", "x", "q"]))
    assert hist == ["1", "2", "2", "3", "x", "q"], hist

run_tests("F", [
    ("solve_all_circuits (+ energy check)", _test_f1, 1),
    ("circuit_table", _test_f2, 1),
    ("run_cli with scripted input", _test_f3, 1),
])

In [ ]:
# Final demo — use this output in your video and slides
def _demo_f():
    circuits = _circuits()
    for cid, c in circuits.items():
        print(cid, {k: round(v, 4) for k, v in c.node_voltages.items()})
    display(circuit_table(circuits))
    display(circuits["C2"].branch_report().round(6))
    # run_cli(circuits)   # <- uncomment to try the interactive menu

try_demo(_demo_f)

---
## Part T — Your team's `pytest` suite  *(10 pts)*

Move your finished code from this notebook into the `src/voltlab/` modules listed in Milestone 2, then write unit tests in `tests/`.

**Requirements**
* **≥ 15 test functions** (at least 3 written by *each* member — put the author's name in a comment above each test).
* Cover all six Parts, and for each tested function include: a **normal case**, an **edge case** (empty input, zero, boundary value…) and an **error case** (`pytest.raises`).
* At least **2 tests for your OOP classes** must check polymorphism (the same method call behaving differently on two subclasses).
* At least **1 test** must use a small hand-made DataFrame (not the full dataset).
* `pytest -v` must pass from the repository root. Paste the final output in your README.

**Example test file** (`tests/test_basics.py`)
```python
import pytest
from voltlab.basics import parallel_resistance

# Author: <member name>
def test_parallel_resistance_normal_case():
    assert abs(parallel_resistance([100, 100]) - 50) < 1e-9

# Author: <member name>
def test_parallel_resistance_rejects_bad_input():
    with pytest.raises(ValueError):
        parallel_resistance([])
```

**Make `import voltlab` work:** add a `pyproject.toml` (or run tests with `PYTHONPATH=src pytest -v`).

**Extra credit EC3 (+5 pts): Continuous Integration.** Add `.github/workflows/tests.yml` so GitHub runs your tests on every push:
```yaml
name: tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "3.12"}
      - run: pip install -r requirements.txt
      - run: PYTHONPATH=src pytest -v
```
A green ✅ badge in your README looks great to recruiters.

## Part Q — Code quality & Git history  *(10 pts)*
* Every function, class and method has a **docstring** (purpose, parameters, return value).
* Meaningful variable names, no copy-pasted blocks, no unused code, comments for non-obvious steps.
* Work merged through **Pull Requests**, each reviewed by a teammate.
* Every member has **≥ 10 meaningful commits** (`git shortlog -sn --all`). Members with very few commits may receive a reduced individual grade.

## Milestone 3 submission checklist
- [ ] *Kernel → Restart & Run All* runs without errors.
- [ ] All autograder tests pass (see the summary cell below).
- [ ] Code moved into `src/voltlab/`; `pytest -v` passes with ≥ 15 tests.
- [ ] README explains how to install, run, and test the project.
- [ ] Create a GitHub **release** with the tag **`v1.0-m3`** (*Releases → Draft a new release*) and submit the release link on the LMS.

In [ ]:
# ===================== AUTOGRADER SUMMARY =====================
def autograder_summary():
    """Print the score of every Part that has been tested so far."""
    total_e = total_p = 0
    print(f"{'Part':<6}{'Earned':>8}{'Possible':>10}")
    for part in ["A", "B", "C", "D", "E", "F"]:
        if part in _SCORES:
            e, p = _SCORES[part]
            total_e += e
            total_p += p
            print(f"{part:<6}{e:>8}{p:>10}")
    print(f"{'Total':<6}{total_e:>8}{total_p:>10}   (autograded part of Milestone 3, out of 35)")
    if "EC" in _SCORES:
        e, p = _SCORES["EC"]
        print(f"{'EC':<6}{e:>8}{p:>10}   (autograded extra credit EC1 + EC2, out of 10)")
    return total_e, total_p


_ = autograder_summary()

---
# Milestone 4 — Project video & final presentation  *(due Thu, Dec 10, 2026 · 30 pts)*

### 4.1 Project video — **8 to 10 minutes** (15 pts)
Record a screen-capture video (Zoom, OBS Studio, PowerPoint *Record*, Loom…) that explains and demonstrates your project. **Every member must appear on camera or speak for at least 1 minute.** Videos shorter than 8:00 or longer than 10:30 lose 2 points per started minute outside the range.

| Time | Segment | Suggested presenter |
|---|---|---|
| 0:00 – 1:00 | Team introduction, the engineering problem, why it matters | Project Manager |
| 1:00 – 2:30 | The dataset and the Pandas pipeline (loading, cleaning, aggregation) | Data Engineer |
| 2:30 – 4:00 | Numerical methods with NumPy — show one equation and the code that implements it | Numerical Methods Lead |
| 4:00 – 5:30 | OOP design — walk through the UML class diagram (inheritance, composition, polymorphism) | Software Architect |
| 5:30 – 7:30 | **Live demo** — run the CLI, produce the report and plots | Project Manager + any member |
| 7:30 – 8:45 | Testing — run `pytest -v`, explain one edge-case test, show the Git history / PRs | Quality & Docs Lead |
| 8:45 – 10:00 | Results, limitations, what you would add next, what each member learned | Everyone |

**Technical requirements:** 1080p (or 720p minimum), readable code (font ≥ 14 pt), clear audio. Upload to YouTube (*Unlisted*), OneDrive or the LMS, and put the link in your README.

### 4.2 Final presentation file (10 pts)
A slide deck of **10–15 slides**, saved as `docs/presentation.pdf` (or `.pptx`) in your repo and uploaded to the LMS. Required slides:
1. Title — project name, team number, member names & roles
2. Problem statement & motivation
3. Dataset description (columns, size, data-quality issues found)
4. System architecture — how the modules (`basics`, `numerics`, `data_pipeline`, `models`, `cli`) fit together
5. UML class diagram
6. Key algorithm(s) — equations + short code snippet
7. Pandas pipeline — before/after cleaning statistics
8. Results — at least **2 plots** produced by your code, with interpretation
9. Testing strategy — number of tests, types of tests, coverage of edge cases
10. Live-demo screenshot (CLI output / report)
11. Challenges & how you solved them
12. Future work
13. Individual contributions (who did what — must match the Git history)
14. *(Optional)* Extra-credit features you completed (EC1–EC4)

### 4.3 Peer evaluation (5 pts)
Each member submits the confidential peer-evaluation form on the LMS rating every teammate (including themself) on contribution, reliability and communication. Individual grades may be adjusted when the peer evaluation and Git history show unequal contribution.

### Final submission checklist (Dec 10)
- [ ] Video link (8–10 min) in README and on the LMS
- [ ] `docs/presentation.pdf` (or `.pptx`) pushed and uploaded
- [ ] README completed: description, features, install/run/test instructions, screenshots of plots, `pytest` output, video link, team & roles
- [ ] Final release tag **`v2.0-final`** created on GitHub
- [ ] Peer evaluation submitted by **each** member

---
# ⭐ Extra Credit — up to +20 points  *(optional · due Thu, Dec 10, 2026)*

Extra credit lets strong teams push the project further — and gives you more to talk about in interviews. **Start only after Parts A–F pass** (extra credit counts only when you have at least 25 of the 35 Milestone 3 autograder points). Each item is worth **5 points**; you may do any combination.

| Item | What to do | How it is graded | Pts |
|---|---|---|---|
| **EC1 — Dashboard** | Implement the 2 × 2 Matplotlib dashboard described below. | Autograder check (a `Figure` with 4 titled sub-plots) + plot quality in your video | +5 |
| **EC2 — Advanced engineering feature** | Implement the project-specific feature described below. | Autograder | +5 |
| **EC3 — Continuous Integration** | Add the GitHub Actions workflow from *Part T* so `pytest` runs on every push; add the status badge to your README. | Green ✅ run on the final commit | +5 |
| **EC4 — Go further** *(choose one)* | **(a)** Run your pipeline on a **real public dataset** (suggestions in *"Put this project on your CV"*), **(b)** build an **interactive app** (`ipywidgets` in the notebook or a `streamlit` web app), or **(c)** make the repo an **installable package** (`pyproject.toml`, `pip install -e .`, and a command `python -m voltlab.cli` that starts your menu). | Documented in the README and demonstrated for ≥ 30 s in the video | +5 |

**Matplotlib reminder for EC1** — a 2 × 2 grid of plots:
```python
fig, axes = plt.subplots(2, 2, figsize=(13, 8))   # axes is a 2 x 2 array of Axes
axes[0, 0].plot(x, y, label="data")               # top-left
axes[0, 0].set(title="My title", xlabel="x unit", ylabel="y unit")
axes[0, 0].legend()
axes[1, 1].bar(names, values)                     # bottom-right
fig.tight_layout()                                # stop labels overlapping
return fig                                        # return the Figure (do not call plt.show() inside)
```

## EC1 — VoltLab dashboard  *(+5)*
Implement `plot_dashboard(circuits)` returning a 2 × 2 `Figure`:

| Position | Plot | Hint |
|---|---|---|
| top-left | node voltage vs. node number for every circuit | loop over `circuits.items()`, `axes[0, 0].plot(nodes, volts, "o-", label=cid)` |
| top-right | bar chart of the power absorbed by each component of C2 | `circuits["C2"].branch_report()` → `axes[0, 1].bar(rep["name"], rep["power_w"] * 1000)` (mW) |
| bottom-left | histogram of 5,000 `monte_carlo_divider(12, 10e3, 4.7e3, 5)` samples with dashed lines at ±2 % of nominal | `axes[1, 0].hist(s, bins=40)`, `axes[1, 0].axvline(x, ls="--")` |
| bottom-right | RC step response (1 kΩ, 1 µF, 5 V) for 0–5 ms with τ marked | `rc_step_response`, `axvline(1.0)` (x axis in ms) |

## EC2 — DC sweep and whole-circuit Monte Carlo  *(+5)*
* **`dc_sweep(circuit, source_name, values)`** *(2 pts)* — for each value: set `circuit[source_name].value`, call `circuit.solve()` and record a row `{"value": v, "V1": …, "V2": …}` (one column per node 1 … `n_nodes`). Return a DataFrame. **Restore the original source value and re-solve at the end** (a `try/finally` block guarantees this even if something fails).
* **`tolerance_analysis(circuit, node, n=1000, seed=0)`** *(3 pts)* — `rng = np.random.default_rng(seed)`; remember every resistor's nominal value; for each of `n` trials set every `Resistor` to `nominal × (1 + rng.uniform(−tol, +tol))` with `tol = tolerance_pct / 100`, solve, and store `V(node)`. Restore all nominal values and re-solve at the end. Return the NumPy array of samples. *(Combine it with your Part D `monte_carlo_table` to report the manufacturing yield!)*

In [ ]:
def plot_dashboard(circuits):
    """Draw the 2 x 2 VoltLab dashboard and return the Figure."""
    # HINT ── see the EC1 table: fig, axes = plt.subplots(2, 2, figsize=(13, 8)) ... fig.tight_layout(); return fig
    # YOUR CODE HERE
    raise NotImplementedError()


def dc_sweep(circuit, source_name, values):
    """Solve the circuit for every source value; return a DataFrame of node voltages."""
    # HINT ── source = circuit[source_name]; original = source.value; rows = []
    #   try:
    #       for v in values: source.value = float(v); volts = circuit.solve()
    #                        rows.append({"value": float(v), **{f"V{k}": volts[k] for k in range(1, circuit.n_nodes + 1)}})
    #   finally: source.value = original; circuit.solve()
    #   return pd.DataFrame(rows)
    # YOUR CODE HERE
    raise NotImplementedError()


def tolerance_analysis(circuit, node, n=1000, seed=0):
    """Return n Monte Carlo samples of V(node) with every resistor varied within its tolerance."""
    # HINT ── follow the EC2 description; use a try/finally like dc_sweep so the nominal values are always restored.
    #   resistors = [c for c in circuit.components.values() if isinstance(c, Resistor)]
    #   nominal = {r.name: r.value for r in resistors}; samples = np.empty(n)
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
def _test_ec1():
    fig = plot_dashboard(_circuits())
    assert isinstance(fig, plt.Figure), "return the Matplotlib Figure"
    assert len(fig.axes) >= 4 and all(ax.get_title() for ax in fig.axes[:4]), "4 sub-plots, each with a title"
    plt.close(fig)

def _test_ec2a():
    c = _divider()
    sw = dc_sweep(c, "V1", [0, 5, 10])
    assert list(sw.columns) == ["value", "V1", "V2"]
    assert_close(sw["V2"].to_numpy(), np.array([0, 5, 10]) * 4.7 / 14.7)
    assert c["V1"].value == 12.0, "original source value must be restored"

def _test_ec2b():
    c1 = _circuits()["C1"]
    nominal = c1.voltage(2)
    s = tolerance_analysis(c1, 2, n=400, seed=3)
    assert isinstance(s, np.ndarray) and s.shape == (400,) and s.std() > 0
    assert abs(s.mean() - nominal) < 0.02
    assert_close(c1["R1"].value, 10e3); assert_close(c1.voltage(2), nominal)

run_tests("EC", [
    ("EC1 plot_dashboard returns a 2 x 2 titled Figure", _test_ec1, 5),
    ("EC2 dc_sweep restores the source", _test_ec2a, 2),
    ("EC2 tolerance_analysis restores nominal values", _test_ec2b, 3),
])
_ = autograder_summary()

In [ ]:
# Extra-credit demo
def _demo_ec():
    circuits = _circuits()
    plot_dashboard(circuits)
    plt.show()
    c1 = circuits["C1"]
    v = c1.voltage(2)
    display(monte_carlo_table({"C1 V2 (whole circuit)": tolerance_analysis(c1, 2, n=2000)}, 0.98 * v, 1.02 * v).round(4))
    display(dc_sweep(c1, "V1", np.linspace(0, 15, 6)).round(3))

try_demo(_demo_ec)

---
# Grading rubric — 100 points + 20 extra credit = 120 points possible

| Milestone | Component | Criteria | Pts |
|---|---|---|---|
| **M1** | Groups | 5 members, all roles assigned, `check_team()` passes, LMS form submitted on time | 5 |
| **M2** | GitHub | Repo named correctly, all members **and instructor** added, required structure pushed, workflow agreed | 10 |
| **M3** | Part A — Python basics & functions | Autograder | 5 |
| | Part B — Loops & decision making | Autograder (no NumPy/Pandas in Part B) | 6 |
| | Part C — NumPy | Autograder + vectorized code | 7 |
| | Part D — Pandas | Autograder | 7 |
| | Part E — OOP | Autograder + sound class design (ABC, inheritance, composition, polymorphism) | 7 |
| | Part F — Integration & CLI | Autograder + working report and menu | 3 |
| | Part T — Team `pytest` suite | ≥ 15 meaningful tests, normal/edge/error cases, passes | 10 |
| | Part Q — Code quality & Git | Docstrings, readability, PRs, ≥ 10 commits per member | 10 |
| **M4** | Video (8–10 min) | Clear explanation, live demo, every member speaks, within time | 15 |
| | Presentation file | All required slides, clear plots, consistent with code | 10 |
| | Peer evaluation | Submitted by every member | 5 |
| | | **Required total** | **100** |
| **EC** | EC1 — Dashboard | `plot_dashboard` autograder check + readable, labelled plots shown in the video | +5 |
| | EC2 — Advanced engineering feature | Autograder | +5 |
| | EC3 — GitHub Actions CI | Workflow file present, green ✅ run on the final commit, badge in README | +5 |
| | EC4 — Go further | One option from the EC4 list, documented in README and shown in the video | +5 |
| | | **Maximum with extra credit** | **120** |

*Extra credit is only counted when the required work of Milestone 3 is substantially complete (at least 25 of the 35 autograder points). Late policy and academic-integrity rules follow the course syllabus. Using AI assistants is allowed only as described in the syllabus, and every team member must be able to explain every line of the code in the video and when asked.*

---
# Put this project on your CV

**Suggested résumé entry** (edit the numbers to match *your* results):

> **VoltLab — DC Circuit Simulator in Python** · *Python, NumPy, Pandas, Matplotlib, pytest, Git* · Nov–Dec 2026
> * Developed an object-oriented circuit simulator that parses SPICE-style netlists and solves resistive networks with Modified Nodal Analysis (`numpy.linalg.solve`), verified against hand calculations and energy conservation.
> * Built a Pandas pipeline that cleans netlists with SI prefixes and inconsistent formatting, decodes resistor colour codes, and checks component availability against lab inventory.
> * Implemented Monte Carlo tolerance analysis predicting __ % manufacturing yield for a ±2 % output specification; delivered with a CLI and __ unit tests by a 5-person team using GitHub pull requests.

**Stretch goals** (optional, great interview talking points):
* Add `Capacitor` and `Inductor` classes and a transient solver using backward-Euler companion models.
* Add a voltage-controlled voltage source (op-amp model) and simulate an inverting amplifier.
* Read real LTspice `.cir` netlists and compare your node voltages with LTspice results.
* Use `scipy.sparse` to solve a 1,000-node resistor mesh and measure the speed-up.

### How to make the repository CV-ready (after final grades are posted)
1. **Make the repo public** (*Settings → General → Danger Zone → Change visibility*). Each member can also **fork** it to their own account and **pin** it on their GitHub profile.
2. **Polish the README** — a one-sentence pitch at the top, a screenshot/GIF of the plots, a "Features" list, "Quick start" (`pip install -r requirements.txt`, `python -m voltlab.cli`), test badge, video link, and a "Team" section with each member's role.
3. **Add the project to LinkedIn** (*Profile → Add section → Projects*), link the repo and the video, and tag your teammates.
4. **Prepare a 60-second interview story** using the STAR format: *Situation* (the engineering problem), *Task* (your role), *Action* (what you built — name a specific algorithm or class), *Result* (numbers: tests passing, data size, accuracy, speed-up).
5. Under **Skills** on your résumé list only what you actually used: *Python, NumPy, Pandas, Matplotlib, OOP, pytest, Git/GitHub, GitHub Actions (if you did the bonus)*.

---
*End of project notebook — good luck, and build something you are proud to show!*